# CityLens learned pipeline in Colab

Runs everything in the cloud: clone repo, keep the **dataset in Google Drive** (download once, reuse forever), install dependencies, bootstrap the learned-model package into Colab, and run the **Prithvi + street-view + fusion** experiments.

---

## What to do now

1. **Upload this notebook to Colab**.
2. Choose **T4 GPU** in `Runtime -> Change runtime type`.
3. **Run all**.
4. **First run:** Section 3 mounts Google Drive and downloads/extracts CityLens once to **My Drive -> GeoBench -> CityLens-Data**.
5. **Next runs:** Run all again. The notebook reuses the Drive copy and the learned experiments resume from checkpoints under `Results/global_learned/`.

This notebook does **not** require Gemini, OpenAI, or any other API key.

**Drive check:** After section 3, open [drive.google.com](https://drive.google.com) -> **My Drive -> GeoBench -> CityLens-Data**. You should see the dataset folders and a `Results/` folder after training starts.

## 1. Clone CityLens repo

In [30]:
!git clone --depth 1 https://github.com/tsinghua-fib-lab/CityLens.git
%cd CityLens

Cloning into 'CityLens'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 84 (delta 26), reused 36 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 2.03 MiB | 1.30 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/CityLens/CityLens


In [31]:
# Install learned-model deps
# We force a consistent scientific stack, then restart the runtime once.
# This avoids numpy / scikit-learn binary mismatches in Colab.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.citylens_learned_deps_ready_v2")
if not MARKER.exists():
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.13.1",
        "pandas==2.2.2",
        "scikit-learn==1.5.2",
        "terratorch",
        "timm",
        "peft",
        "captum",
        "safetensors",
    ]
    print("Installing Colab-safe learned-model dependencies...")
    subprocess.check_call(cmd)
    MARKER.write_text("ok", encoding="utf-8")
    print("Dependencies installed. Restarting runtime once to avoid binary mismatches...")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already prepared for this runtime. Continuing.")

Dependencies already prepared for this runtime. Continuing.


## 2. Add path config and patch eval code for Colab

In [32]:
import os

# Write into CityLens/ so eval finds it when run from /content/CityLens
target_dir = "CityLens" if os.path.isdir("CityLens") else "."
path_config_file = os.path.join(target_dir, "path_config.py")
PATH_CONFIG = '''"""
Path config for Colab. Data root = /content/CityLens-Data
"""
import os

DATA_ROOT = os.environ.get("CITYLENS_DATA_ROOT", "/content/CityLens-Data")

def benchmark_path(task_name: str, city: str) -> str:
    return os.path.join(DATA_ROOT, "Benchmark", f"{task_name}_{city}.json")

def results_path(task_name: str, city: str, model_name: str, prompt_type: str) -> str:
    safe = model_name.replace("/", "_")
    return os.path.join(DATA_ROOT, "Results", f"{task_name}_{city}_{safe}_{prompt_type}.json")

def summary_csv_path() -> str:
    return os.path.join(DATA_ROOT, "Results", "summary.csv")

def url_file_path() -> str:
    return os.path.join(DATA_ROOT, "Benchmark", "image_urls.csv")
'''

with open(path_config_file, "w") as f:
    f.write(PATH_CONFIG)
print("Wrote", path_config_file)

Wrote ./path_config.py


**Run this next:** Patch config.py so utils.py does not raise ImportError (adds SILICONFLOW_APIKEY, DEEPINFRA_APIKEY).

In [33]:
# No API-key patching needed for the learned-model pipeline
print("Skipping config.py API-key patch: this notebook does not use Gemini/OpenAI.")

Skipping config.py API-key patch: this notebook does not use Gemini/OpenAI.


In [34]:
# Patch global_indicator.py to use path_config
import os
if os.path.isdir("CityLens"): os.chdir("CityLens")
path = "evaluate/global/global_indicator.py"
with open(path, "r") as f:
    text = f.read()

if "path_config" not in text:
    text = text.replace(
        "from config import STV_NUM, GLOBAL_TASK_NUM\nrandom.seed(42)",
        "from config import STV_NUM, GLOBAL_TASK_NUM\n\ntry:\n    from path_config import benchmark_path, results_path, url_file_path\n    _USE_PATH_CONFIG = True\nexcept ImportError:\n    _USE_PATH_CONFIG = False\n\nrandom.seed(42)"
    )
    text = text.replace(
        '    url_file = ""',
        '    url_file = url_file_path() if _USE_PATH_CONFIG else ""'
    )
    text = text.replace(
        "if city == \"all\":\n        task_path = f\"\"\n    else:\n        task_path = f'\''\n    \n    model_name_full = model_name.replace(\"/\", \"_\")\n    response_path = f'\''",
        "model_name_full = model_name.replace(\"/\", \"_\")\n    if _USE_PATH_CONFIG:\n        task_path = benchmark_path(task_name, city)\n        response_path = results_path(task_name, city, model_name, prompt_type)\n    else:\n        if city == \"all\":\n            task_path = f\"\"\n        else:\n            task_path = f'\''\n        response_path = f'\''"
    )
    import re
    # Replace eval_task path block (allow flexible whitespace)
    pattern = r"(def eval_task\(city, model_name, num_process, prompt_type, task_name\):)\s+if city == \"all\":\s+task_path = f\"\"\s+else:\s+task_path = f'\''\s+model_name_full = model_name\.replace\(\"/\", \"_\"\)\s+response_path = f'\''"
    repl = r"\1\n    model_name_full = model_name.replace(\"/\", \"_\")\n    if _USE_PATH_CONFIG:\n        task_path = benchmark_path(task_name, city)\n        response_path = results_path(task_name, city, model_name, prompt_type)\n    else:\n        if city == \"all\":\n            task_path = f\"\"\n        else:\n            task_path = f'\''\n        response_path = f'\''"
    text = re.sub(pattern, repl, text, count=1)
    with open(path, "w") as f:
        f.write(text)
    print("Patched global_indicator.py")
else:
    print("global_indicator.py already patched")

# Add fallback so task_path/response_path set from env when path_config not loaded
import os
if os.path.isdir("CityLens"): os.chdir("CityLens")
path = "evaluate/global/global_indicator.py"
with open(path, "r") as f: t = f.read()
needle = "    output_dir = os.path.dirname(response_path)"
fallback_insert = "    # Fallback when path_config not loaded (e.g. Colab)\n    if not task_path or not response_path:\n        data_root = os.environ.get(\"CITYLENS_DATA_ROOT\", \"/content/CityLens-Data\")\n        task_path = os.path.join(data_root, \"Benchmark\", f\"{task_name}_{city}.json\")\n        response_path = os.path.join(data_root, \"Results\", f\"{task_name}_{city}_{model_name_full}_{prompt_type}.json\")\n    " + needle
if "if not task_path or not response_path" in t:
    print("eval_task fallback already present.")
elif needle in t:
    before, _, after = t.partition(needle)
    t = before + fallback_insert + after
    t = t.replace("    if not os.path.exists(output_dir):", "    if output_dir and not os.path.exists(output_dir):")
    with open(path, "w") as f: f.write(t)
    print("Added eval_task fallback for Colab.")
else:
    t = t.replace("    if not os.path.exists(output_dir):", "    if output_dir and not os.path.exists(output_dir):")
    with open(path, "w") as f: f.write(t)
    print("Patched makedirs guard. Set CITYLENS_DATA_ROOT.")

Patched global_indicator.py
Added eval_task fallback for Colab.


In [35]:
# Patch metrics.py to use path_config
path = "evaluate/global/metrics.py"
with open(path, "r") as f:
    text = f.read()

if "path_config" not in text:
    text = text.replace(
        "from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\n",
        "from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\n\ntry:\n    from path_config import results_path, summary_csv_path\n    _USE_PATH_CONFIG = True\nexcept ImportError:\n    _USE_PATH_CONFIG = False\n"
    )
    old = """    args = parser.parse_args()
    model_name_full = args.model_name.replace("/", "_")
    response_path = f''
    summary_path = f''"""
    new = """    args = parser.parse_args()
    model_name_full = args.model_name.replace("/", "_")
    if _USE_PATH_CONFIG:
        response_path = results_path(args.task_name, args.city_name, args.model_name, args.prompt_type)
        summary_path = summary_csv_path()
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
    else:
        response_path = f''
        summary_path = f''"""
    text = text.replace(old, new)
    with open(path, "w") as f:
        f.write(text)
    print("Patched metrics.py")
else:
    pass
text = open(path).read()
if "if not response_path:" not in text:
    old_block = """        summary_path = f''
    y_true, y_pred"""
    new_block = """        summary_path = f''
    if not response_path:
        data_root = os.environ.get("CITYLENS_DATA_ROOT", "")
        if data_root:
            response_path = os.path.join(data_root, "Results", f"{args.task_name}_{args.city_name}_{model_name_full}_{args.prompt_type}.json")
            summary_path = os.path.join(data_root, "Results", "summary.csv")
            os.makedirs(os.path.dirname(summary_path), exist_ok=True)
    y_true, y_pred"""
    text = text.replace(old_block, new_block)
    with open(path, "w") as f: f.write(text)
    print("Added metrics env fallback.")
else:
    print("metrics.py already patched")

Patched metrics.py
Added metrics env fallback.


### 2b. API-free notebook

The old Gemini/GDP baseline cells were removed. This notebook is now fully focused on the **learned-model pipeline** and does **not require any API key**.

In [36]:
# Legacy API adapter removed
print("This notebook does not use any LLM API adapters or API keys.")

This notebook does not use any LLM API adapters or API keys.


In [37]:
# Legacy API patching removed
print("Skipping legacy API utils patch: no API keys needed.")

Skipping legacy API utils patch: no API keys needed.


## 3. Google Drive + CityLens dataset (~10 GB)

**Dataset is stored in your Google Drive** so you only download once. Each run: mount Drive → reuse existing data (no re-download). You need **~12 GB free** in Drive. The first run downloads and extracts to `My Drive/GeoBench/CityLens-Data/`. **To use a different Google account** (e.g. one with more space), run the cell below first, then re-run the next cell.

**If the Drive link doesn’t let you choose another account**, Colab is reusing the account you’re already signed into. Use one of these:

1. **Incognito window:** Close this tab. Open a **new incognito/private window** (Ctrl+Shift+N / Cmd+Shift+N). Go to [colab.research.google.com](https://colab.research.google.com), open this notebook. Run the unmount cell, then the Drive cell. When the link appears, sign in with the account that has enough space.

2. **Sign out first:** In this same browser go to [accounts.google.com](https://accounts.google.com) → **Sign out**. Then back in Colab run the unmount cell, then the Drive cell and click the link — you’ll get the full “Choose an account” screen.

3. **Different browser or profile:** Open Colab in another browser (or another Chrome profile) where you’re only logged into the account you want for Drive.

In [38]:
# SWITCH DRIVE ACCOUNT: Run this cell first, then run the next cell.
# When the "Connect to Google Drive" link appears, click it → choose "Use another account" → sign in with the account that has enough space.
'''from google.colab import drive
drive.flush_and_unmount()
print("Drive unmounted. Now run the cell BELOW. When the link appears, click it and pick your other account.")'''

'from google.colab import drive\ndrive.flush_and_unmount()\nprint("Drive unmounted. Now run the cell BELOW. When the link appears, click it and pick your other account.")'

In [39]:
import os
from google.colab import drive

mountpoint = "/content/drive"

if os.path.ismount(mountpoint):
    print("Drive already mounted at", mountpoint)
else:
    if os.path.exists(mountpoint) and os.listdir(mountpoint):
        print("Removing stale mountpoint contents:", mountpoint)
        !rm -rf /content/drive
    drive.mount(mountpoint)

print("Done.")

Drive already mounted at /content/drive
Done.


In [40]:
# Data root is set by the cell above (from Google Drive). This just confirms.
import os
from pathlib import Path
data_root = os.environ.get("CITYLENS_DATA_ROOT")
if data_root and Path(data_root).exists():
    print("Data root OK (from Drive):", data_root)
else:
    print("CITYLENS_DATA_ROOT not set or path missing. Re-run the 'Google Drive + CityLens dataset' cell above.")

CITYLENS_DATA_ROOT not set or path missing. Re-run the 'Google Drive + CityLens dataset' cell above.


In [41]:
# Verify: data is stored in Google Drive (persists across sessions)
import os
root = os.environ.get("CITYLENS_DATA_ROOT", "")
if root.startswith("/content/drive"):
    drive_path = root.replace("/content/drive/MyDrive/", "My Drive/")
    print("Saved in Drive:", drive_path)
    print("Next session: run this notebook again; section 3 will mount Drive and reuse this data (no re-download).")
else:
    print("CITYLENS_DATA_ROOT:", root or "(not set)")

CITYLENS_DATA_ROOT: (not set)


### Verify dataset (run anytime)

Run this cell to check whether the dataset exists in Colab and where it is. If you see "NOT FOUND", re-run the **Download CityLens-Data** cell (section 3); Colab's `/content/` is wiped when the runtime restarts.

In [42]:
import os
from pathlib import Path

print("=== Dataset verification ===\n")

# Prefer CITYLENS_DATA_ROOT (set by section 3 from Drive)
data_root = os.environ.get("CITYLENS_DATA_ROOT")
if data_root and Path(data_root).exists():
    if (Path(data_root) / "Benchmark").exists() and (Path(data_root) / "Dataset").exists():
        print("✓ Data root (from Drive):", data_root)
    else:
        print("CITYLENS_DATA_ROOT set but Benchmark/Dataset missing at:", data_root)
        data_root = None

# Else check Google Drive path
if not data_root:
    drive_base = Path("/content/drive/MyDrive/GeoBench/CityLens-Data")
    if drive_base.exists():
        if (drive_base / "Benchmark").exists() and (drive_base / "Dataset").exists():
            data_root = str(drive_base)
            print("✓ Data root (Drive):", data_root)
        else:
            for d in drive_base.iterdir():
                if d.is_dir() and (d / "Benchmark").exists() and (d / "Dataset").exists():
                    data_root = str(d)
                    print("✓ Data root (Drive):", data_root)
                    break
    if not data_root and drive_base.exists():
        print("Drive folder exists but Benchmark/Dataset not found. Subdirs:", [x.name for x in drive_base.iterdir() if x.is_dir()][:10])

# Else check /content (legacy)
if not data_root:
    base = Path("/content/CityLens-Data")
    if base.exists():
        if (base / "Benchmark").exists() and (base / "Dataset").exists():
            data_root = str(base)
        else:
            for d in base.iterdir():
                if d.is_dir() and (d / "Benchmark").exists() and (d / "Dataset").exists():
                    data_root = str(d)
                    break
    if data_root:
        print("✓ Data root (/content):", data_root)

if data_root:
    print("\nCITYLENS_DATA_ROOT =", repr(data_root))
else:
    print("\n✗ No valid data root. Run section 3 (Google Drive + CityLens dataset) first.")

=== Dataset verification ===

✓ Data root (Drive): /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data

CITYLENS_DATA_ROOT = '/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data'


### Debug: why "0 valid data" in metrics?

If the **Run metrics** cell prints **length of vals: 0 0** or **No valid data found**, run the cell below. It inspects the results JSON (reference/response) and shows why numbers might not be extracted. Common causes: (1) Results file empty → re-run GDP eval. (2) Wrong keys in JSON → eval wrote different format. (3) Model returned verbose text → `extract_float` looks for the first number in text; if none, that item is skipped.

In [43]:
# Legacy API debug removed
print("Skipping legacy API debug: no API keys are required.")
print("Watch the Section 6 training logs and saved metrics instead.")

Skipping legacy API debug: no API keys are required.
Watch the Section 6 training logs and saved metrics instead.


## 4. Install dependencies

In [44]:
"""
Paste this into a Colab cell and run it. No requirements.txt needed.
Uses torch>=2.2 for Colab compatibility; skips triton and nvidia-* (Colab has its own CUDA).
"""
PACKAGES = [
    "addict==2.4.0",
    "affine==2.4.0",
    "aiohappyeyeballs==2.4.0",
    "aiohttp==3.10.5",
    "aiomultiprocess==0.9.0",
    "aiosignal==1.3.1",
    "annotated-types==0.7.0",
    "anyio==4.5.0",
    "asttokens==2.4.1",
    "async-timeout==4.0.3",
    "attrs==24.2.0",
    "beautifulsoup4==4.12.3",
    "bs4==0.0.2",
    "certifi==2024.8.30",
    "charset-normalizer==3.3.2",
    "click==8.1.7",
    "click-plugins==1.1.1",
    "cligj==0.7.2",
    "comm==0.2.2",
    "contextily==1.6.2",
    "contourpy==1.3.0",
    "coord-convert==0.2.1",
    "coverage==7.6.1",
    "crcmod==1.7",
    "cryptography==42.0.5",
    "cycler==0.12.1",
    "dashscope==1.20.10",
    "debugpy==1.8.5",
    "decorator==5.1.1",
    "distro==1.9.0",
    "dnspython==2.6.1",
    "exceptiongroup==1.2.2",
    "executing==2.1.0",
    "fastapi==0.115.0",
    "filelock==3.16.1",
    "fiona==1.10.1",
    "fonttools==4.53.1",
    "frozenlist==1.4.1",
    "fsspec==2024.9.0",
    "ftfy==6.2.3",
    "gast==0.5.4",
    "geographiclib==2.0",
    "geojson==3.1.0",
    "geopandas==0.14.3",
    "geopy==2.4.1",
    "grpcio==1.66.1",
    "grpcio-tools==1.62.3",
    "h11==0.14.0",
    "httpcore==1.0.5",
    "httpx==0.27.2",
    "huggingface-hub==0.25.0",
    "idna==3.10",
    "igraph==0.11.6",
    "importlib_metadata==7.1.0",
    "iniconfig==2.0.0",
    "ipykernel==6.29.5",
    "ipython==8.27.0",
    "jedi==0.19.1",
    "Jinja2==3.1.4",
    "jiter==0.5.0",
    "jmespath==0.10.0",
    "joblib==1.4.2",
    "json_repair==0.25.3",
    "jsonlines==4.0.0",
    "jupyter_client==8.6.3",
    "jupyter_core==5.7.2",
    "kiwisolver==1.4.7",
    "Levenshtein==0.25.1",
    "loguru==0.7.2",
    "lru-dict==1.3.0",
    "lxml==5.3.0",
    "MarkupSafe==2.1.5",
    "matplotlib==3.8.3",
    "matplotlib-inline==0.1.7",
    "mercantile==1.2.1",
    "modelscope==1.13.3",
    "mosstool==0.2.34",
    "mpmath==1.3.0",
    "multidict==6.1.0",
    "nest-asyncio==1.6.0",
    "networkx==3.3",
    "numpy==1.26.4",
    "open_clip_torch==2.26.1",
    "openai==1.46.0",
    "opencv-python==4.10.0.84",
    "packaging==24.1",
    "pandas==2.2.2",
    "parso==0.8.4",
    "path4gmns==0.9.8",
    "peft==0.12.0",
    "pexpect==4.9.0",
    "pillow==10.4.0",
    "platformdirs==4.3.6",
    "pluggy==1.5.0",
    "prompt_toolkit==3.0.47",
    "protobuf==4.25.5",
    "psutil==6.0.0",
    "ptyprocess==0.7.0",
    "pure_eval==0.2.3",
    "pyarrow==16.0.0",
    "pyarrow-hotfix==0.6",
    "pydantic==2.9.2",
    "pydantic_core==2.23.4",
    "pydeck==0.8.0",
    "Pygments==2.18.0",
    "pymongo==4.9.1",
    "pyparsing==3.1.4",
    "pyproj==3.6.1",
    "pytest==8.3.3",
    "pytest-cov==5.0.0",
    "pytest-cover==3.0.0",
    "pytest-coverage==0.0",
    "python-dateutil==2.9.0.post0",
    "python-moss==0.4.4",
    "pytz==2024.2",
    "PyYAML==6.0.2",
    "pyzmq==26.2.0",
    "rapidfuzz==3.9.7",
    "rasterio==1.3.9",
    "regex==2024.9.11",
    "requests==2.32.3",
    "safetensors==0.4.5",
    "scikit-learn==1.5.2",
    "scipy==1.13.0",
    "seaborn==0.13.2",
    "setproctitle==1.3.3",
    "shapely==2.0.6",
    "simplejson==3.19.2",
    "six==1.16.0",
    "sniffio==1.3.1",
    "snuggs==1.4.7",
    "sortedcontainers==2.4.0",
    "soupsieve==2.6",
    "sse-starlette==2.1.3",
    "stack-data==0.6.3",
    "starlette==0.38.5",
    "stringcase==1.2.0",
    "sympy==1.13.3",
    "tenacity==8.5.0",
    "texttable==1.7.0",
    "thefuzz==0.22.1",
    "threadpoolctl==3.5.0",
    "timm==1.0.9",
    "tokenizers==0.19.1",
    "tomli==2.0.1",
    "tomlkit==0.12.0",
    "torch>=2.2.0",
    "torchvision>=0.17.0",
    "tornado==6.4.1",
    "tqdm==4.66.5",
    "traitlets==5.14.3",
    "transformers==4.41.2",
    "trl==0.9.6",
    "types-protobuf==4.25.0.20240417",
    "typing_extensions==4.12.2",
    "tzdata==2024.1",
    "urllib3==2.2.3",
    "uvicorn==0.30.6",
    "wcwidth==0.2.13",
    "websocket-client==1.8.0",
    "xxhash==3.4.1",
    "xyzservices==2024.9.0",
    "yapf==0.40.2",
    "yarl==1.11.1",
    "zipp==3.18.1",
]

# Skip install if key deps already present (safe for Run All)
import subprocess
import sys
_skip = False
try:
    import torch
    import openai
    import google.generativeai
    _skip = True
except ImportError:
    pass
if _skip:
    print("Key deps already installed (skipping pip).")
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + PACKAGES
    subprocess.check_call(cmd)
    print("Requirements installed.")


Key deps already installed (skipping pip).


In [45]:
# Skip if gdal already available (safe for Run All)
try:
    import osgeo
    print("GDAL already installed.")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdal"])


GDAL already installed.


## 5. Legacy API baseline removed

This notebook no longer runs any Gemini/OpenAI baseline and **does not ask for API keys**.

Use **Section 6 (Learned global-task pipeline)** for training, evaluation, checkpointing, and explainability.

In [46]:
# API-free notebook check
# CITYLENS_DATA_ROOT is set in Section 3.
import os
print("Skipping API key setup: this notebook is API-free.")
print("CITYLENS_DATA_ROOT =", os.environ.get("CITYLENS_DATA_ROOT"))

Skipping API key setup: this notebook is API-free.
CITYLENS_DATA_ROOT = None


In [47]:
# Legacy API-driven evaluation removed
print("Skipping legacy Gemini/OpenAI evaluation.")
print("Use Section 6 (Learned global-task pipeline) instead.")


Skipping legacy Gemini/OpenAI evaluation.
Use Section 6 (Learned global-task pipeline) instead.


In [48]:
# Legacy API metrics removed
print("Skipping legacy API metrics.")
print("Use the saved metrics under CITYLENS_DATA_ROOT/Results/global_learned/ instead.")

"""
def _extract_float(v):
    if v is None: return None
    if isinstance(v, (float, int)): return float(v)
    if isinstance(v, str):
        s = v.replace(',', '').replace('$', '').replace(chr(163), '').strip()
        if len(s) <= 100:
            try: return float(s)
            except ValueError: pass
        m = _re.search(r'-?[\\d,]+\\.?\\d*', v)
        if m: return float(m.group().replace(',', ''))
    return None
y_true, y_pred = [], []
for _it in _jd:
    ref = _extract_float(_it.get('reference'))
    rsp = _it.get('response')
    if rsp is not None and not isinstance(rsp, str): rsp = str(rsp)
    pred = _extract_float(rsp)
    if ref is not None and pred is not None:
        y_true.append(ref); y_pred.append(pred)
print('length of vals:', len(y_true), len(y_pred))
if len(y_true) == 0 and _jd:
    _it = _jd[0]
    print('DEBUG first item keys:', list(_it.keys()))
    print('  reference', type(_it.get('reference')).__name__, repr(str(_it.get('reference'))[:150]))
    print('  response', type(_it.get('response')).__name__, repr(str(_it.get('response'))[:150]))
    print('  _extract_float(ref)=', _extract_float(_it.get('reference')), '_extract_float(resp)=', _extract_float(_it.get('response')))
if len(y_true) > 0:
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    mse, mae, r2 = mean_squared_error(y_true, y_pred), mean_absolute_error(y_true, y_pred), r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    print('MSE', mse, 'MAE', mae, 'R2', r2, 'RMSE', rmse)
    os.makedirs(os.path.dirname(_summ), exist_ok=True)
    _write_header = not os.path.exists(_summ)
    with open(_summ, 'a', newline='') as _f:
        _w = _csv.DictWriter(_f, fieldnames=['city','model','prompt_type','MSE','MAE','R2','RMSE'])
        if _write_header: _w.writeheader()
        _w.writerow({'city':'all','model':'gemini-1.5-flash','prompt_type':'simple','MSE':mse,'MAE':mae,'R2':r2,'RMSE':rmse})
    print('Metrics written to', _summ)
else:
    print('No valid data found.')
print('Metrics done.')
path = '/content/CityLens/evaluate/global/metrics.py'
with open(path) as f:
    t = f.read()
import re as _re
if 'match = re.search' not in t and 'def extract_float' in t:
    _start = t.find('def extract_float(value):')
    _end = t.find('def process_json(', _start)
    if _start != -1 and _end != -1:
        _new_ef = '''def extract_float(value):
    if isinstance(value, (float, int)):
        return float(value)
    if isinstance(value, str):
        value_str = value.replace(\",\", \"\").replace(\"$\", \"\").replace(\"£\", \"\").strip()
        if len(value_str) <= 100:
            try:
                return float(value_str)
            except ValueError:
                pass
        match = re.search(r\"-?[\\d,]+\\.?\\d*\", value)
        if match:
            try:
                return float(match.group().replace(\",\", \"\"))
            except ValueError:
                pass
    return None

'''
        t = t[:_start] + _new_ef + t[_end:]
        print('Patched extract_float for long/verbose responses.')
    else:
        pass
start = t.find('args = parser.parse_args()')
end_line = t.find('y_true, y_pred = process_json(response_path, args.prompt_type)')
if start != -1 and end_line != -1:
    line_start = t.rfind(chr(10), 0, start) + 1
    indent = t[line_start:start]
    if indent.strip() or len(indent) > 12:
        indent = '    '
    METRICS_MAIN = indent + "args = parser.parse_args()\n" + indent + "model_name_full = args.model_name.replace(\"/\", \"_\")\n" + indent + "data_root = os.environ.get(\"CITYLENS_DATA_ROOT\", \"/content/CityLens-Data/CityLens-data\")\n" + indent + "response_path = os.path.join(data_root, \"Results\", f\"{args.task_name}_{args.city_name}_{model_name_full}_{args.prompt_type}.json\")\n" + indent + "summary_path = os.path.join(data_root, \"Results\", \"summary.csv\")\n" + indent + "os.makedirs(os.path.dirname(summary_path), exist_ok=True)\n" + indent + "y_true, y_pred = process_json(response_path, args.prompt_type)\n"
    end = t.find(chr(10), end_line)
    end = (end + 1) if end != -1 else len(t)
    t = t[:line_start] + METRICS_MAIN + t[end:]
    with open(path, 'w') as f:
        f.write(t)
    print('Replaced metrics path block (env-based).')
"""


Skipping legacy API metrics.
Use the saved metrics under CITYLENS_DATA_ROOT/Results/global_learned/ instead.


'\ndef _extract_float(v):\n    if v is None: return None\n    if isinstance(v, (float, int)): return float(v)\n    if isinstance(v, str):\n        s = v.replace(\',\', \'\').replace(\'$\', \'\').replace(chr(163), \'\').strip()\n        if len(s) <= 100:\n            try: return float(s)\n            except ValueError: pass\n        m = _re.search(r\'-?[\\d,]+\\.?\\d*\', v)\n        if m: return float(m.group().replace(\',\', \'\'))\n    return None\ny_true, y_pred = [], []\nfor _it in _jd:\n    ref = _extract_float(_it.get(\'reference\'))\n    rsp = _it.get(\'response\')\n    if rsp is not None and not isinstance(rsp, str): rsp = str(rsp)\n    pred = _extract_float(rsp)\n    if ref is not None and pred is not None:\n        y_true.append(ref); y_pred.append(pred)\nprint(\'length of vals:\', len(y_true), len(y_pred))\nif len(y_true) == 0 and _jd:\n    _it = _jd[0]\n    print(\'DEBUG first item keys:\', list(_it.keys()))\n    print(\'  reference\', type(_it.get(\'reference\')).__name__, 

## 6. (Optional) Copy results to Drive

```python
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/CityLens-Data/Results /content/drive/MyDrive/
```

## 6. Learned global-task pipeline: Prithvi + street-view + fusion

This section replaces the old API baseline with a **learned-model pipeline** for the 5 global CityLens tasks:
`gdp`, `pop`, `acc2health`, `carbon`, `build_height`.

Implemented experiment families:
- **Low-cost control**: frozen street-view embeddings + `LassoCV`
- **Satellite-only**: `prithvi_rgb_lora`, `dinov2_sat`, `resnet50_sat`
- **Street-view-only**: `resnet50`, `clip_vitb16`, `dinov2_vitb14`, optional `swin_t`
- **Fusion**: `late` or `gated`
- **Explainability**: integrated gradients, leave-one-view-out, modality ablation

Important architectural note:
- Official **Prithvi-EO-2.0-300M** is pretrained on **6-band HLS** (not CityLens RGB PNGs).
- This notebook uses an explicit **RGB -> 6-channel adapter** and treats Prithvi as an **adapted baseline**, not a native HLS setup.

Checkpointing / resume:
- All learned jobs write to `CITYLENS_DATA_ROOT/Results/global_learned/...`
- Each experiment saves `config.json`, `history.csv` or regression artifacts, `metrics.json`, `val_predictions.csv`, and checkpoint files
- The notebook runs with `--skip_if_done --resume` where applicable, so **Run all** is safe after Colab restarts.

### 6a. First-run restart note

On the **first** Section 6 run in a fresh Colab runtime, the dependency cell may intentionally **restart the runtime once** after installing a consistent `numpy/scipy/pandas/scikit-learn` stack.

If that happens, just **Run all** again from the top. The marker file under `/content/` prevents repeating the reinstall loop in the same runtime session.

In [49]:
# Bootstrap learned-model package into the Colab CityLens repo
from pathlib import Path

REPO_ROOT = Path("/content/CityLens")
PKG_ROOT = REPO_ROOT / "evaluate" / "global_learned"
PKG_ROOT.mkdir(parents=True, exist_ok=True)

FILES = {
    "evaluate/global_learned/__init__.py": r'''from .data import GLOBAL_TASKS, load_global_items, resolve_data_root

__all__ = ["GLOBAL_TASKS", "load_global_items", "resolve_data_root"]
''',
    "evaluate/global_learned/utils.py": r'''import csv
import json
import random
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List

import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


GLOBAL_TASKS = ["gdp", "pop", "acc2health", "carbon", "build_height"]


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def slugify(text: str) -> str:
    return (
        text.replace("/", "_")
        .replace(" ", "_")
        .replace(":", "_")
        .replace(",", "_")
        .replace("__", "_")
        .strip("_")
    )


def results_root(data_root: Path) -> Path:
    return data_root / "Results" / "global_learned"


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


@dataclass
class ExperimentConfig:
    task_name: str
    branch: str
    satellite_model: str
    street_model: str
    fusion_type: str
    pooling: str
    image_size: int
    batch_size: int
    epochs: int
    lr: float
    weight_decay: float
    val_frac: float
    seed: int
    lora_r: int
    num_workers: int

    def experiment_name(self) -> str:
        bits = [
            self.task_name,
            self.branch,
            self.satellite_model,
            self.street_model,
            self.fusion_type,
            self.pooling,
            f"bs{self.batch_size}",
            f"ep{self.epochs}",
            f"lr{self.lr}",
            f"seed{self.seed}",
        ]
        return slugify("-".join([b for b in bits if b and b != "none"]))


def compute_metrics(y_true: Iterable[float], y_pred: Iterable[float]) -> Dict[str, float]:
    y_true = np.asarray(list(y_true), dtype=np.float32)
    y_pred = np.asarray(list(y_pred), dtype=np.float32)
    if len(y_true) == 0:
        return {"mse": float("nan"), "mae": float("nan"), "r2": float("nan"), "rmse": float("nan")}
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    return {"mse": float(mse), "mae": float(mae), "r2": float(r2), "rmse": float(rmse)}


def save_json(path: Path, payload: Dict) -> None:
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


def append_csv(path: Path, row: Dict, fieldnames: List[str]) -> None:
    ensure_dir(path.parent)
    write_header = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def append_experiment_log(data_root: Path, config: ExperimentConfig, split_path: Path, checkpoint_dir: Path, metrics: Dict[str, float], status: str) -> None:
    row = {
        "timestamp": datetime.utcnow().isoformat(timespec="seconds"),
        "task_name": config.task_name,
        "branch": config.branch,
        "satellite_model": config.satellite_model,
        "street_model": config.street_model,
        "fusion_type": config.fusion_type,
        "pooling": config.pooling,
        "image_size": config.image_size,
        "batch_size": config.batch_size,
        "epochs": config.epochs,
        "lr": config.lr,
        "weight_decay": config.weight_decay,
        "val_frac": config.val_frac,
        "seed": config.seed,
        "lora_r": config.lora_r,
        "split_path": str(split_path),
        "checkpoint_dir": str(checkpoint_dir),
        "status": status,
        **metrics,
    }
    append_csv(results_root(data_root) / "experiment_log.csv", row, fieldnames=list(row.keys()))


def dump_config(path: Path, config: ExperimentConfig) -> None:
    save_json(path, asdict(config))
''',
    "evaluate/global_learned/data.py": r'''from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

from .utils import GLOBAL_TASKS, ensure_dir, save_json


def resolve_data_root(root: Optional[str] = None) -> Path:
    if root:
        return Path(root)
    env_root = os.environ.get("CITYLENS_DATA_ROOT")
    if env_root:
        return Path(env_root)
    try:
        from path_config import DATA_ROOT  # type: ignore
        return Path(DATA_ROOT)
    except Exception as exc:
        raise RuntimeError("CITYLENS_DATA_ROOT is not set and path_config is unavailable") from exc


def global_task_candidates(task_name: str) -> List[str]:
    if task_name not in GLOBAL_TASKS:
        raise ValueError(f"Unsupported global task: {task_name}")
    return [
        f"Benchmark/{task_name}_all.json",
        f"Benchmark/all_global_{task_name}_task.json",
        f"Dataset/all_global_{task_name}_task_all.json",
        f"Dataset/all_global_{task_name}_task-all.json",
    ]


def resolve_task_json(data_root: Path, task_name: str) -> Path:
    for rel in global_task_candidates(task_name):
        p = data_root / rel
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find task JSON for {task_name} under {data_root}")


def build_image_index(root: Path, folder_name: str) -> Dict[str, Path]:
    folder = root / folder_name
    index: Dict[str, Path] = {}
    if not folder.exists():
        return index
    for dirpath, _, files in os.walk(folder):
        for name in files:
            if name not in index:
                index[name] = Path(dirpath) / name
    return index


def resolve_image_path(image_path: str, data_root: Path, image_index: Optional[Dict[str, Path]] = None, default_subdir: Optional[str] = None) -> Path:
    p = Path(image_path)
    if p.is_file():
        return p
    candidates = [data_root / image_path.lstrip("/"), data_root / p.name]
    if default_subdir:
        candidates.append(data_root / default_subdir / p.name)
    for cand in candidates:
        if cand.is_file():
            return cand
    if image_index and p.name in image_index:
        return image_index[p.name]
    raise FileNotFoundError(f"Could not resolve image path: {image_path}")


def normalize_global_item(item: Dict, data_root: Path, sat_index: Dict[str, Path], stv_index: Dict[str, Path]) -> Dict:
    images = item.get("images", [])
    if not images:
        raise ValueError("Benchmark item has no images")
    sat = resolve_image_path(images[0], data_root, sat_index, "satellite_image")
    stv = []
    for path in images[1:]:
        try:
            stv.append(resolve_image_path(path, data_root, stv_index, "street_view_image"))
        except FileNotFoundError:
            continue
    return {
        "id": item.get("area") or item.get("img_name") or item.get("ct") or sat.stem,
        "satellite": sat,
        "street_views": stv,
        "reference": float(item["reference"]),
        "reference_normalized": item.get("reference_normalized"),
        "prompt": item.get("prompt", ""),
        "raw": item,
    }


def load_global_items(task_name: str, data_root: Path) -> List[Dict]:
    task_json = resolve_task_json(data_root, task_name)
    items = json.load(open(task_json, "r", encoding="utf-8"))
    sat_index = build_image_index(data_root, "satellite_image")
    stv_index = build_image_index(data_root, "street_view_image")
    normalized = []
    for item in items:
        try:
            normalized.append(normalize_global_item(item, data_root, sat_index, stv_index))
        except Exception:
            continue
    if not normalized:
        raise RuntimeError(f"No usable items found for {task_name} from {task_json}")
    return normalized


def make_or_load_split(items: Sequence[Dict], split_path: Path, seed: int = 42, val_frac: float = 0.1) -> Tuple[List[Dict], List[Dict]]:
    ensure_dir(split_path.parent)
    if split_path.exists():
        split = json.load(open(split_path, "r", encoding="utf-8"))
        id_map = {item["id"]: item for item in items}
        train_items = [id_map[i] for i in split["train_ids"] if i in id_map]
        val_items = [id_map[i] for i in split["val_ids"] if i in id_map]
        return train_items, val_items
    ids = [item["id"] for item in items]
    random.Random(seed).shuffle(ids)
    n_val = max(1, int(len(ids) * val_frac))
    val_ids = ids[:n_val]
    train_ids = ids[n_val:]
    save_json(split_path, {"seed": seed, "val_frac": val_frac, "train_ids": train_ids, "val_ids": val_ids})
    id_map = {item["id"]: item for item in items}
    return [id_map[i] for i in train_ids], [id_map[i] for i in val_ids]


def make_rgb_transform(image_size: int, train: bool) -> transforms.Compose:
    ops: List[object] = [transforms.Resize((image_size, image_size))]
    if train:
        ops += [transforms.RandomHorizontalFlip()]
    ops += [
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(ops)


class GlobalSatelliteDataset(Dataset):
    def __init__(self, items: Sequence[Dict], image_size: int = 224, train: bool = True):
        self.items = list(items)
        self.transform = make_rgb_transform(image_size, train)

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        item = self.items[idx]
        image = Image.open(item["satellite"]).convert("RGB")
        return {"image": self.transform(image), "target": torch.tensor(item["reference"], dtype=torch.float32), "id": item["id"]}


class GlobalStreetViewDataset(Dataset):
    def __init__(self, items: Sequence[Dict], image_size: int = 224, train: bool = True, max_views: int = 10):
        self.items = [x for x in items if x["street_views"]]
        self.transform = make_rgb_transform(image_size, train)
        self.max_views = max_views
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.items)

    def _load_views(self, paths: Sequence[Path]) -> Tuple[torch.Tensor, torch.Tensor]:
        images = []
        mask = []
        for path in list(paths)[: self.max_views]:
            image = Image.open(path).convert("RGB")
            images.append(self.transform(image))
            mask.append(1.0)
        while len(images) < self.max_views:
            images.append(torch.zeros_like(images[0]) if images else torch.zeros(3, self.image_size, self.image_size))
            mask.append(0.0)
        return torch.stack(images, dim=0), torch.tensor(mask, dtype=torch.float32)

    def __getitem__(self, idx: int):
        item = self.items[idx]
        views, mask = self._load_views(item["street_views"])
        return {"street_views": views, "street_mask": mask, "target": torch.tensor(item["reference"], dtype=torch.float32), "id": item["id"]}


class GlobalFusionDataset(Dataset):
    def __init__(self, items: Sequence[Dict], image_size: int = 224, train: bool = True, max_views: int = 10):
        self.items = [x for x in items if x["street_views"]]
        self.sat_transform = make_rgb_transform(image_size, train)
        self.stv_transform = make_rgb_transform(image_size, train)
        self.max_views = max_views
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        item = self.items[idx]
        sat = self.sat_transform(Image.open(item["satellite"]).convert("RGB"))
        street = []
        mask = []
        for path in list(item["street_views"])[: self.max_views]:
            street.append(self.stv_transform(Image.open(path).convert("RGB")))
            mask.append(1.0)
        while len(street) < self.max_views:
            street.append(torch.zeros(3, self.image_size, self.image_size))
            mask.append(0.0)
        return {
            "image": sat,
            "street_views": torch.stack(street, dim=0),
            "street_mask": torch.tensor(mask, dtype=torch.float32),
            "target": torch.tensor(item["reference"], dtype=torch.float32),
            "id": item["id"],
        }
''',
    "evaluate/global_learned/models.py": r'''from __future__ import annotations

from typing import Optional

import timm
import torch
import torch.nn as nn


def reduce_backbone_output(out: torch.Tensor | dict | list | tuple) -> torch.Tensor:
    if isinstance(out, dict):
        for key in ["features", "x", "encoder_output", "last_hidden_state"]:
            if key in out:
                out = out[key]
                break
        else:
            out = next(iter(out.values()))
    if isinstance(out, (list, tuple)):
        out = out[-1]
    if out.dim() == 5:
        return out.mean(dim=(2, 3, 4))
    if out.dim() == 4:
        return out.mean(dim=(2, 3))
    if out.dim() == 3:
        return out.mean(dim=1)
    if out.dim() == 2:
        return out
    return out.view(out.size(0), -1)


class TimmEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.model_name = model_name
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        feat = reduce_backbone_output(out)
        if self.feature_dim is None:
            self.feature_dim = feat.size(-1)
        return feat


class PrithviRGBEncoder(nn.Module):
    def __init__(self, backbone_name: str = "prithvi_eo_v2_300", lora_r: int = 8):
        super().__init__()
        from terratorch.registry import BACKBONE_REGISTRY

        self.input_adapter = nn.Conv2d(3, 6, kernel_size=1, bias=False)
        self.backbone = BACKBONE_REGISTRY.build(
            backbone_name,
            pretrained=True,
            bands=["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"],
            num_frames=1,
        )
        self.feature_dim: Optional[int] = None
        self.lora_applied = False
        if lora_r > 0:
            try:
                from peft import LoraConfig, get_peft_model
                cfg = LoraConfig(
                    r=lora_r,
                    lora_alpha=max(8, lora_r * 2),
                    lora_dropout=0.05,
                    bias="none",
                    target_modules=["qkv", "q_proj", "k_proj", "v_proj", "proj"],
                )
                self.backbone = get_peft_model(self.backbone, cfg)
                self.lora_applied = True
            except Exception:
                self.lora_applied = False

    def _forward_backbone(self, x: torch.Tensor):
        attempts = [x, x.unsqueeze(2), x.unsqueeze(1)]
        last_err: Optional[Exception] = None
        for candidate in attempts:
            try:
                return self.backbone(candidate)
            except Exception as exc:
                last_err = exc
        raise RuntimeError(f"Prithvi forward failed for all input layouts. Last error: {last_err}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_adapter(x)
        out = self._forward_backbone(x)
        feat = reduce_backbone_output(out)
        if self.feature_dim is None:
            self.feature_dim = feat.size(-1)
        return feat


class AttentionPool(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        logits = self.score(x).squeeze(-1)
        if mask is not None:
            logits = logits.masked_fill(mask <= 0, -1e9)
        weights = torch.softmax(logits, dim=-1)
        return (x * weights.unsqueeze(-1)).sum(dim=1)


class StreetViewRegressor(nn.Module):
    def __init__(self, encoder: nn.Module, pooling: str = "mean", hidden_dim: int = 256):
        super().__init__()
        self.encoder = encoder
        self.pooling_name = pooling
        self.pool: Optional[AttentionPool] = None
        self.head: Optional[nn.Sequential] = None
        self.hidden_dim = hidden_dim

    def _pool_views(self, feats: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
        if self.pooling_name == "mean":
            if mask is None:
                return feats.mean(dim=1)
            denom = mask.sum(dim=1, keepdim=True).clamp_min(1.0)
            return (feats * mask.unsqueeze(-1)).sum(dim=1) / denom
        if self.pool is None:
            self.pool = AttentionPool(feats.size(-1)).to(feats.device)
        return self.pool(feats, mask)

    def forward_features(self, street_views: torch.Tensor, street_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        b, v, c, h, w = street_views.shape
        feats = self.encoder(street_views.view(b * v, c, h, w))
        feats = feats.view(b, v, -1)
        return self._pool_views(feats, street_mask)

    def forward(self, street_views: torch.Tensor, street_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        feat = self.forward_features(street_views, street_mask)
        if self.head is None:
            self.head = nn.Sequential(nn.Linear(feat.size(-1), self.hidden_dim), nn.ReLU(), nn.Linear(self.hidden_dim, 1)).to(feat.device)
        return self.head(feat).squeeze(-1)


class SatelliteRegressor(nn.Module):
    def __init__(self, encoder: nn.Module, hidden_dim: int = 256):
        super().__init__()
        self.encoder = encoder
        self.hidden_dim = hidden_dim
        self.head: Optional[nn.Sequential] = None

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.forward_features(x)
        if self.head is None:
            self.head = nn.Sequential(nn.Linear(feat.size(-1), self.hidden_dim), nn.ReLU(), nn.Linear(self.hidden_dim, 1)).to(feat.device)
        return self.head(feat).squeeze(-1)


class FusionRegressor(nn.Module):
    def __init__(self, sat_encoder: nn.Module, street_encoder: nn.Module, pooling: str = "mean", fusion_type: str = "late", hidden_dim: int = 256):
        super().__init__()
        self.sat_model = SatelliteRegressor(sat_encoder, hidden_dim=hidden_dim)
        self.street_model = StreetViewRegressor(street_encoder, pooling=pooling, hidden_dim=hidden_dim)
        self.fusion_type = fusion_type
        self.gate: Optional[nn.Linear] = None
        self.head: Optional[nn.Sequential] = None
        self.hidden_dim = hidden_dim

    def forward(self, image: torch.Tensor, street_views: torch.Tensor, street_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        sat_feat = self.sat_model.forward_features(image)
        stv_feat = self.street_model.forward_features(street_views, street_mask)
        if self.fusion_type == "gated":
            if self.gate is None:
                self.gate = nn.Linear(sat_feat.size(-1) + stv_feat.size(-1), 2).to(sat_feat.device)
            fused = torch.cat([sat_feat, stv_feat], dim=-1)
            weights = torch.softmax(self.gate(fused), dim=-1)
            min_dim = min(sat_feat.size(-1), stv_feat.size(-1))
            sat_crop = sat_feat[:, :min_dim]
            stv_crop = stv_feat[:, :min_dim]
            combined = weights[:, :1] * sat_crop + weights[:, 1:] * stv_crop
        else:
            combined = torch.cat([sat_feat, stv_feat], dim=-1)
        if self.head is None:
            self.head = nn.Sequential(nn.Linear(combined.size(-1), self.hidden_dim), nn.ReLU(), nn.Linear(self.hidden_dim, 1)).to(combined.device)
        return self.head(combined).squeeze(-1)


def make_satellite_encoder(name: str, lora_r: int = 8) -> nn.Module:
    if name == "prithvi_rgb_lora":
        return PrithviRGBEncoder("prithvi_eo_v2_300", lora_r=lora_r)
    if name == "prithvi_rgb_lora_tl":
        return PrithviRGBEncoder("prithvi_eo_v2_300_tl", lora_r=lora_r)
    if name == "resnet50_sat":
        return TimmEncoder("resnet50")
    if name == "dinov2_sat":
        return TimmEncoder("vit_base_patch14_dinov2.lvd142m")
    raise ValueError(f"Unknown satellite model: {name}")


def make_street_encoder(name: str) -> nn.Module:
    if name == "resnet50":
        return TimmEncoder("resnet50")
    if name == "clip_vitb16":
        return TimmEncoder("vit_base_patch16_clip_224.openai")
    if name == "dinov2_vitb14":
        return TimmEncoder("vit_base_patch14_dinov2.lvd142m")
    if name == "swin_t":
        return TimmEncoder("swin_tiny_patch4_window7_224")
    raise ValueError(f"Unknown street model: {name}")
''',
    "evaluate/global_learned/train.py": r'''from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data import GlobalFusionDataset, GlobalSatelliteDataset, GlobalStreetViewDataset, load_global_items, make_or_load_split, resolve_data_root
from .models import FusionRegressor, SatelliteRegressor, StreetViewRegressor, make_satellite_encoder, make_street_encoder
from .utils import ExperimentConfig, GLOBAL_TASKS, append_experiment_log, compute_metrics, dump_config, ensure_dir, results_root, save_json, set_seed


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="CityLens learned global-task baselines")
    parser.add_argument("--task_name", default="all", choices=["all"] + GLOBAL_TASKS)
    parser.add_argument("--branch", default="satellite", choices=["satellite", "street", "fusion"])
    parser.add_argument("--satellite_model", default="prithvi_rgb_lora")
    parser.add_argument("--street_model", default="clip_vitb16")
    parser.add_argument("--fusion_type", default="late", choices=["late", "gated"])
    parser.add_argument("--pooling", default="mean", choices=["mean", "attention"])
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--weight_decay", type=float, default=1e-2)
    parser.add_argument("--val_frac", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--data_root", default=None)
    parser.add_argument("--skip_if_done", action="store_true")
    parser.add_argument("--resume", action="store_true")
    return parser.parse_args()


def build_dataloaders(branch: str, train_items: List[Dict], val_items: List[Dict], image_size: int, batch_size: int, num_workers: int) -> Tuple[DataLoader, DataLoader]:
    if branch == "satellite":
        train_ds = GlobalSatelliteDataset(train_items, image_size=image_size, train=True)
        val_ds = GlobalSatelliteDataset(val_items, image_size=image_size, train=False)
    elif branch == "street":
        train_ds = GlobalStreetViewDataset(train_items, image_size=image_size, train=True)
        val_ds = GlobalStreetViewDataset(val_items, image_size=image_size, train=False)
    else:
        train_ds = GlobalFusionDataset(train_items, image_size=image_size, train=True)
        val_ds = GlobalFusionDataset(val_items, image_size=image_size, train=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader


def build_model(cfg: ExperimentConfig) -> nn.Module:
    if cfg.branch == "satellite":
        return SatelliteRegressor(make_satellite_encoder(cfg.satellite_model, lora_r=cfg.lora_r))
    if cfg.branch == "street":
        return StreetViewRegressor(make_street_encoder(cfg.street_model), pooling=cfg.pooling)
    return FusionRegressor(make_satellite_encoder(cfg.satellite_model, lora_r=cfg.lora_r), make_street_encoder(cfg.street_model), pooling=cfg.pooling, fusion_type=cfg.fusion_type)


def forward_batch(model: nn.Module, batch: Dict, branch: str, device: torch.device):
    target = batch["target"].to(device)
    if branch == "satellite":
        pred = model(batch["image"].to(device))
    elif branch == "street":
        pred = model(batch["street_views"].to(device), batch["street_mask"].to(device))
    else:
        pred = model(batch["image"].to(device), batch["street_views"].to(device), batch["street_mask"].to(device))
    return pred, target


def evaluate(model: nn.Module, loader: DataLoader, branch: str, device: torch.device):
    model.eval()
    y_true, y_pred, predictions = [], [], []
    with torch.no_grad():
        for batch in loader:
            pred, target = forward_batch(model, batch, branch, device)
            pred_cpu = pred.detach().cpu().tolist()
            target_cpu = target.detach().cpu().tolist()
            ids = batch["id"]
            y_true.extend(target_cpu)
            y_pred.extend(pred_cpu)
            predictions.extend({"id": sample_id, "target": float(t), "prediction": float(p)} for sample_id, t, p in zip(ids, target_cpu, pred_cpu))
    return compute_metrics(y_true, y_pred), predictions


def save_predictions(path: Path, rows: List[Dict]) -> None:
    ensure_dir(path.parent)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "target", "prediction"])
        writer.writeheader()
        writer.writerows(rows)


def run_task(task_name: str, args: argparse.Namespace, data_root: Path) -> None:
    cfg = ExperimentConfig(
        task_name=task_name,
        branch=args.branch,
        satellite_model=args.satellite_model,
        street_model=args.street_model,
        fusion_type=args.fusion_type if args.branch == "fusion" else "none",
        pooling=args.pooling if args.branch in {"street", "fusion"} else "none",
        image_size=args.image_size,
        batch_size=args.batch_size,
        epochs=args.epochs,
        lr=args.lr,
        weight_decay=args.weight_decay,
        val_frac=args.val_frac,
        seed=args.seed,
        lora_r=args.lora_r,
        num_workers=args.num_workers,
    )
    task_root = results_root(data_root) / task_name / cfg.experiment_name()
    ckpt_dir = ensure_dir(task_root / "checkpoints")
    split_path = results_root(data_root) / "splits" / f"{task_name}_seed{args.seed}_val{args.val_frac}.json"
    best_ckpt = ckpt_dir / "best.pt"
    last_ckpt = ckpt_dir / "last.pt"
    metrics_path = task_root / "metrics.json"
    preds_path = task_root / "val_predictions.csv"
    history_path = task_root / "history.csv"
    if args.skip_if_done and best_ckpt.exists() and metrics_path.exists():
        metrics = json.load(open(metrics_path, "r", encoding="utf-8"))
        print(f"[skip] {task_name}: found existing checkpoint at {best_ckpt}")
        append_experiment_log(data_root, cfg, split_path, ckpt_dir, metrics, status="skipped")
        return
    items = load_global_items(task_name, data_root)
    train_items, val_items = make_or_load_split(items, split_path, seed=args.seed, val_frac=args.val_frac)
    train_loader, val_loader = build_dataloaders(cfg.branch, train_items, val_items, args.image_size, args.batch_size, args.num_workers)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    loss_fn = nn.MSELoss()
    start_epoch = 1
    best_rmse = float("inf")
    history_rows: List[Dict] = []
    if args.resume and last_ckpt.exists():
        state = torch.load(last_ckpt, map_location=device)
        model.load_state_dict(state["model"], strict=False)
        optimizer.load_state_dict(state["optimizer"])
        start_epoch = int(state["epoch"]) + 1
        best_rmse = float(state.get("best_rmse", best_rmse))
        print(f"[resume] {task_name}: resuming from epoch {start_epoch}")
    dump_config(task_root / "config.json", cfg)
    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        running = 0.0
        seen = 0
        prog = tqdm(train_loader, desc=f"{task_name} epoch {epoch}/{args.epochs}", leave=False)
        for batch in prog:
            pred, target = forward_batch(model, batch, cfg.branch, device)
            loss = loss_fn(pred, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            batch_n = target.numel()
            running += loss.item() * batch_n
            seen += batch_n
            prog.set_postfix(train_loss=running / max(1, seen))
        val_metrics, predictions = evaluate(model, val_loader, cfg.branch, device)
        row = {"epoch": epoch, "train_loss": running / max(1, seen), **val_metrics}
        history_rows.append(row)
        if val_metrics["rmse"] < best_rmse:
            best_rmse = val_metrics["rmse"]
            torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "best_rmse": best_rmse, "config": cfg.__dict__}, best_ckpt)
            save_predictions(preds_path, predictions)
            save_json(metrics_path, {"best_epoch": epoch, **val_metrics})
        torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "best_rmse": best_rmse, "config": cfg.__dict__}, last_ckpt)
        with open(history_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "mse", "mae", "r2", "rmse"])
            writer.writeheader()
            writer.writerows(history_rows)
        print(f"[{task_name}] epoch={epoch} val_rmse={val_metrics['rmse']:.4f} val_r2={val_metrics['r2']:.4f}")
    final_metrics = json.load(open(metrics_path, "r", encoding="utf-8")) if metrics_path.exists() else {}
    append_experiment_log(data_root, cfg, split_path, ckpt_dir, final_metrics, status="finished")


def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    data_root = resolve_data_root(args.data_root)
    tasks = GLOBAL_TASKS if args.task_name == "all" else [args.task_name]
    for task_name in tasks:
        run_task(task_name, args, data_root)


if __name__ == "__main__":
    main()
''',
    "evaluate/global_learned/explain.py": r'''from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
from captum.attr import IntegratedGradients

from .data import GlobalFusionDataset, GlobalSatelliteDataset, GlobalStreetViewDataset, load_global_items, make_or_load_split, resolve_data_root
from .models import FusionRegressor, SatelliteRegressor, StreetViewRegressor, make_satellite_encoder, make_street_encoder
from .utils import GLOBAL_TASKS, ensure_dir, results_root, save_json, set_seed


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Explain learned CityLens global-task models")
    parser.add_argument("--task_name", required=True, choices=GLOBAL_TASKS)
    parser.add_argument("--branch", required=True, choices=["satellite", "street", "fusion"])
    parser.add_argument("--checkpoint_dir", required=True)
    parser.add_argument("--satellite_model", default="prithvi_rgb_lora")
    parser.add_argument("--street_model", default="clip_vitb16")
    parser.add_argument("--fusion_type", default="late", choices=["late", "gated"])
    parser.add_argument("--pooling", default="mean", choices=["mean", "attention"])
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--data_root", default=None)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--val_frac", type=float, default=0.1)
    return parser.parse_args()


def build_model(args: argparse.Namespace) -> torch.nn.Module:
    if args.branch == "satellite":
        return SatelliteRegressor(make_satellite_encoder(args.satellite_model))
    if args.branch == "street":
        return StreetViewRegressor(make_street_encoder(args.street_model), pooling=args.pooling)
    return FusionRegressor(make_satellite_encoder(args.satellite_model), make_street_encoder(args.street_model), pooling=args.pooling, fusion_type=args.fusion_type)


def load_val_sample(args: argparse.Namespace, data_root: Path) -> Dict:
    items = load_global_items(args.task_name, data_root)
    split_path = results_root(data_root) / "splits" / f"{args.task_name}_seed{args.seed}_val{args.val_frac}.json"
    _, val_items = make_or_load_split(items, split_path, seed=args.seed, val_frac=args.val_frac)
    if args.branch == "satellite":
        ds = GlobalSatelliteDataset(val_items, image_size=args.image_size, train=False)
    elif args.branch == "street":
        ds = GlobalStreetViewDataset(val_items, image_size=args.image_size, train=False)
    else:
        ds = GlobalFusionDataset(val_items, image_size=args.image_size, train=False)
    return ds[0]


def satellite_integrated_gradients(model: torch.nn.Module, sample: Dict, device: torch.device) -> np.ndarray:
    model.eval()
    x = sample["image"].unsqueeze(0).to(device)
    ig = IntegratedGradients(model)
    attr = ig.attribute(x, target=None)
    return attr.detach().cpu().numpy()[0]


def street_leave_one_view_out(model: torch.nn.Module, sample: Dict, device: torch.device) -> List[Dict]:
    model.eval()
    x = sample["street_views"].unsqueeze(0).to(device)
    mask = sample["street_mask"].unsqueeze(0).to(device)
    with torch.no_grad():
        base = model(x, mask).item()
    drops = []
    for i in range(x.size(1)):
        if mask[0, i].item() <= 0:
            continue
        x_mod = x.clone()
        mask_mod = mask.clone()
        mask_mod[0, i] = 0
        x_mod[0, i] = 0
        with torch.no_grad():
            pred = model(x_mod, mask_mod).item()
        drops.append({"view_index": i, "base_prediction": base, "ablated_prediction": pred, "delta": base - pred})
    return drops


def fusion_modality_ablation(model: torch.nn.Module, sample: Dict, device: torch.device) -> Dict:
    model.eval()
    image = sample["image"].unsqueeze(0).to(device)
    views = sample["street_views"].unsqueeze(0).to(device)
    mask = sample["street_mask"].unsqueeze(0).to(device)
    with torch.no_grad():
        full = model(image, views, mask).item()
        sat_zero = model(torch.zeros_like(image), views, mask).item()
        stv_zero = model(image, torch.zeros_like(views), torch.zeros_like(mask)).item()
    return {
        "full_prediction": full,
        "no_satellite_prediction": sat_zero,
        "no_street_prediction": stv_zero,
        "satellite_contribution_delta": full - sat_zero,
        "street_contribution_delta": full - stv_zero,
    }


def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    data_root = resolve_data_root(args.data_root)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(args).to(device)
    ckpt = torch.load(Path(args.checkpoint_dir) / "checkpoints" / "best.pt", map_location=device)
    model.load_state_dict(ckpt["model"], strict=False)
    sample = load_val_sample(args, data_root)
    out_dir = ensure_dir(Path(args.checkpoint_dir) / "explain")
    if args.branch == "satellite":
        attr = satellite_integrated_gradients(model, sample, device)
        save_json(out_dir / "integrated_gradients_stats.json", {"shape": list(attr.shape), "mean_abs": float(np.abs(attr).mean()), "max_abs": float(np.abs(attr).max())})
    elif args.branch == "street":
        save_json(out_dir / "leave_one_view_out.json", {"rows": street_leave_one_view_out(model, sample, device)})
    else:
        save_json(out_dir / "modality_ablation.json", fusion_modality_ablation(model, sample, device))
    print(f"Saved explainability outputs to {out_dir}")


if __name__ == "__main__":
    main()
''',
}

for rel_path, text in FILES.items():
    out_path = REPO_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(text.rstrip() + "\n", encoding="utf-8")
    print("Wrote", out_path)

print("Bootstrapped learned-model package into", PKG_ROOT)
print("You can now run the Section 6 training cells from this notebook alone.")

Wrote /content/CityLens/evaluate/global_learned/__init__.py
Wrote /content/CityLens/evaluate/global_learned/utils.py
Wrote /content/CityLens/evaluate/global_learned/data.py
Wrote /content/CityLens/evaluate/global_learned/models.py
Wrote /content/CityLens/evaluate/global_learned/train.py
Wrote /content/CityLens/evaluate/global_learned/explain.py
Bootstrapped learned-model package into /content/CityLens/evaluate/global_learned
You can now run the Section 6 training cells from this notebook alone.


In [50]:
# Bootstrap the low-cost feature-control baseline into the Colab CityLens repo
from pathlib import Path

REPO_ROOT = Path("/content/CityLens")
path = REPO_ROOT / "evaluate" / "global_learned" / "feature_control.py"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(r'''from __future__ import annotations

import argparse
import csv
import json
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data import GlobalStreetViewDataset, load_global_items, make_or_load_split, resolve_data_root
from .models import StreetViewRegressor, make_street_encoder
from .utils import ExperimentConfig, GLOBAL_TASKS, append_experiment_log, compute_metrics, dump_config, ensure_dir, results_root, save_json, set_seed


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="CityLens low-cost street-view control")
    parser.add_argument("--task_name", default="all", choices=["all"] + GLOBAL_TASKS)
    parser.add_argument("--street_model", default="resnet50")
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--val_frac", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--data_root", default=None)
    parser.add_argument("--skip_if_done", action="store_true")
    return parser.parse_args()


def make_loader(items: List[Dict], image_size: int, batch_size: int, num_workers: int) -> DataLoader:
    ds = GlobalStreetViewDataset(items, image_size=image_size, train=False)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)


def extract_embeddings(loader: DataLoader, model: StreetViewRegressor, device: torch.device) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    feats, targets, ids = [], [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, leave=False, desc="extract"):
            x = batch["street_views"].to(device)
            mask = batch["street_mask"].to(device)
            emb = model.forward_features(x, mask).detach().cpu().numpy()
            feats.append(emb)
            targets.extend(batch["target"].cpu().numpy().tolist())
            ids.extend(list(batch["id"]))
    return np.concatenate(feats, axis=0), np.asarray(targets, dtype=np.float32), ids


def save_predictions(path: Path, ids: List[str], y_true: np.ndarray, y_pred: np.ndarray) -> None:
    ensure_dir(path.parent)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "target", "prediction"])
        writer.writeheader()
        for sample_id, target, pred in zip(ids, y_true.tolist(), y_pred.tolist()):
            writer.writerow({"id": sample_id, "target": float(target), "prediction": float(pred)})


def run_task(task_name: str, args: argparse.Namespace, data_root: Path) -> None:
    cfg = ExperimentConfig(task_name=task_name, branch="street_feature_control", satellite_model="none", street_model=args.street_model, fusion_type="none", pooling="mean", image_size=args.image_size, batch_size=args.batch_size, epochs=0, lr=0.0, weight_decay=0.0, val_frac=args.val_frac, seed=args.seed, lora_r=0, num_workers=args.num_workers)
    task_root = results_root(data_root) / task_name / cfg.experiment_name()
    ckpt_dir = ensure_dir(task_root / "checkpoints")
    metrics_path = task_root / "metrics.json"
    preds_path = task_root / "val_predictions.csv"
    split_path = results_root(data_root) / "splits" / f"{task_name}_seed{args.seed}_val{args.val_frac}.json"
    artifact_path = ckpt_dir / "lasso.pkl"
    if args.skip_if_done and metrics_path.exists() and artifact_path.exists():
        metrics = json.load(open(metrics_path, "r", encoding="utf-8"))
        append_experiment_log(data_root, cfg, split_path, ckpt_dir, metrics, status="skipped")
        print(f"[skip] {task_name}: feature control already exists")
        return
    items = load_global_items(task_name, data_root)
    train_items, val_items = make_or_load_split(items, split_path, seed=args.seed, val_frac=args.val_frac)
    train_loader = make_loader(train_items, args.image_size, args.batch_size, args.num_workers)
    val_loader = make_loader(val_items, args.image_size, args.batch_size, args.num_workers)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    encoder = make_street_encoder(args.street_model)
    model = StreetViewRegressor(encoder, pooling="mean").to(device)
    x_train, y_train, _ = extract_embeddings(train_loader, model, device)
    x_val, y_val, val_ids = extract_embeddings(val_loader, model, device)
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_val_scaled = scaler.transform(x_val)
    reg = LassoCV(cv=5, random_state=args.seed, max_iter=10000)
    reg.fit(x_train_scaled, y_train)
    y_pred = reg.predict(x_val_scaled)
    metrics = compute_metrics(y_val, y_pred)
    dump_config(task_root / "config.json", cfg)
    save_json(metrics_path, metrics)
    save_predictions(preds_path, val_ids, y_val, y_pred)
    with open(artifact_path, "wb") as f:
        pickle.dump({"scaler": scaler, "regressor": reg}, f)
    append_experiment_log(data_root, cfg, split_path, ckpt_dir, metrics, status="finished")
    print(f"[{task_name}] feature-control rmse={metrics['rmse']:.4f} r2={metrics['r2']:.4f}")


def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    data_root = resolve_data_root(args.data_root)
    tasks = GLOBAL_TASKS if args.task_name == "all" else [args.task_name]
    for task_name in tasks:
        run_task(task_name, args, data_root)


if __name__ == "__main__":
    main()
'''.rstrip() + "\n", encoding="utf-8")
print("Wrote", path)

Wrote /content/CityLens/evaluate/global_learned/feature_control.py


In [51]:
# Refresh bootstrapped learned-model files to the audited versions
from pathlib import Path

REPO_ROOT = Path("/content/CityLens")
UPDATED_FILES = {
    "evaluate/global_learned/utils.py": r'''import csv
import json
import random
from collections import defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Sequence

import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

GLOBAL_TASKS = ["gdp", "pop", "acc2health", "carbon", "build_height"]

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def slugify(text: str) -> str:
    return text.replace("/", "_").replace(" ", "_").replace(":", "_").replace(",", "_").replace("__", "_").strip("_")

def results_root(data_root: Path) -> Path:
    return data_root / "Results" / "global_learned"

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

@dataclass
class ExperimentConfig:
    task_name: str
    branch: str
    satellite_model: str
    street_model: str
    fusion_type: str
    pooling: str
    image_size: int
    batch_size: int
    epochs: int
    lr: float
    weight_decay: float
    val_frac: float
    seed: int
    lora_r: int
    num_workers: int

    def experiment_name(self) -> str:
        bits = [self.task_name, self.branch, self.satellite_model, self.street_model, self.fusion_type, self.pooling, f"bs{self.batch_size}", f"ep{self.epochs}", f"lr{self.lr}", f"seed{self.seed}"]
        return slugify("-".join([b for b in bits if b and b != "none"]))

def compute_metrics(y_true: Iterable[float], y_pred: Iterable[float]) -> Dict[str, float]:
    y_true = np.asarray(list(y_true), dtype=np.float32)
    y_pred = np.asarray(list(y_pred), dtype=np.float32)
    if len(y_true) == 0:
        return {"mse": float("nan"), "mae": float("nan"), "r2": float("nan"), "rmse": float("nan")}
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    return {"mse": float(mse), "mae": float(mae), "r2": float(r2), "rmse": float(rmse)}

def save_json(path: Path, payload: Dict) -> None:
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

def append_csv(path: Path, row: Dict, fieldnames: List[str]) -> None:
    ensure_dir(path.parent)
    write_header = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

def write_rows_csv(path: Path, rows: Sequence[Dict], fieldnames: List[str]) -> None:
    ensure_dir(path.parent)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def append_experiment_log(data_root: Path, config: ExperimentConfig, split_path: Path, checkpoint_dir: Path, metrics: Dict[str, float], status: str) -> None:
    row = {"timestamp": datetime.utcnow().isoformat(timespec="seconds"), "task_name": config.task_name, "branch": config.branch, "satellite_model": config.satellite_model, "street_model": config.street_model, "fusion_type": config.fusion_type, "pooling": config.pooling, "image_size": config.image_size, "batch_size": config.batch_size, "epochs": config.epochs, "lr": config.lr, "weight_decay": config.weight_decay, "val_frac": config.val_frac, "seed": config.seed, "lora_r": config.lora_r, "split_path": str(split_path), "checkpoint_dir": str(checkpoint_dir), "status": status, **metrics}
    append_csv(results_root(data_root) / "experiment_log.csv", row, fieldnames=list(row.keys()))

def dump_config(path: Path, config: ExperimentConfig) -> None:
    save_json(path, asdict(config))

def compute_group_metrics(rows: Sequence[Dict], group_key: str) -> Dict[str, Dict[str, float]]:
    grouped = defaultdict(lambda: {"target": [], "prediction": []})
    for row in rows:
        group = row.get(group_key)
        if group in (None, ""):
            continue
        grouped[str(group)]["target"].append(float(row["target"]))
        grouped[str(group)]["prediction"].append(float(row["prediction"]))
    return {group: compute_metrics(vals["target"], vals["prediction"]) for group, vals in grouped.items()}
''',
    "evaluate/global_learned/data.py": r'''from __future__ import annotations

import json
import os
import random
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

from .utils import GLOBAL_TASKS, ensure_dir, save_json

def resolve_data_root(root: Optional[str] = None) -> Path:
    if root:
        return Path(root)
    env_root = os.environ.get("CITYLENS_DATA_ROOT")
    if env_root:
        return Path(env_root)
    from path_config import DATA_ROOT  # type: ignore
    return Path(DATA_ROOT)

def global_task_candidates(task_name: str) -> List[str]:
    if task_name not in GLOBAL_TASKS:
        raise ValueError(f"Unsupported global task: {task_name}")
    return [f"Benchmark/{task_name}_all.json", f"Benchmark/all_global_{task_name}_task.json", f"Dataset/all_global_{task_name}_task_all.json", f"Dataset/all_global_{task_name}_task-all.json"]

def resolve_task_json(data_root: Path, task_name: str) -> Path:
    for rel in global_task_candidates(task_name):
        p = data_root / rel
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find task JSON for {task_name} under {data_root}")

def build_image_index(root: Path, folder_name: str) -> Dict[str, Path]:
    folder = root / folder_name
    index = {}
    if not folder.exists():
        return index
    for dirpath, _, files in os.walk(folder):
        for name in files:
            if name not in index:
                index[name] = Path(dirpath) / name
    return index

def resolve_image_path(image_path: str, data_root: Path, image_index: Optional[Dict[str, Path]] = None, default_subdir: Optional[str] = None) -> Path:
    p = Path(image_path)
    if p.is_file():
        return p
    candidates = [data_root / image_path.lstrip("/"), data_root / p.name]
    if default_subdir:
        candidates.append(data_root / default_subdir / p.name)
    for cand in candidates:
        if cand.is_file():
            return cand
    if image_index and p.name in image_index:
        return image_index[p.name]
    raise FileNotFoundError(f"Could not resolve image path: {image_path}")

def normalize_global_item(item: Dict, data_root: Path, sat_index: Dict[str, Path], stv_index: Dict[str, Path]) -> Dict:
    images = item.get("images", [])
    if not images:
        raise ValueError("Benchmark item has no images")
    sat = resolve_image_path(images[0], data_root, sat_index, "satellite_image")
    stv = []
    for path in images[1:]:
        try:
            stv.append(resolve_image_path(path, data_root, stv_index, "street_view_image"))
        except FileNotFoundError:
            continue
    return {"id": item.get("area") or item.get("img_name") or item.get("ct") or sat.stem, "city": item.get("city") or item.get("city_name") or sat.parent.name, "satellite": sat, "street_views": stv, "reference": float(item["reference"]), "reference_normalized": item.get("reference_normalized"), "prompt": item.get("prompt", ""), "raw": item}

def load_global_items(task_name: str, data_root: Path, return_report: bool = False):
    task_json = resolve_task_json(data_root, task_name)
    items = json.load(open(task_json, "r", encoding="utf-8"))
    sat_index = build_image_index(data_root, "satellite_image")
    stv_index = build_image_index(data_root, "street_view_image")
    normalized, skipped = [], Counter()
    for item in items:
        try:
            normalized.append(normalize_global_item(item, data_root, sat_index, stv_index))
        except Exception as exc:
            skipped[type(exc).__name__] += 1
    if not normalized:
        raise RuntimeError(f"No usable items found for {task_name} from {task_json}")
    report = {"task_name": task_name, "task_json": str(task_json), "total_records": len(items), "usable_records": len(normalized), "skipped_records": int(sum(skipped.values())), "skip_reasons": dict(skipped), "records_with_street_views": sum(1 for item in normalized if item["street_views"]), "records_without_street_views": sum(1 for item in normalized if not item["street_views"])}
    if return_report:
        return normalized, report
    return normalized

def filter_items_for_branch(items: Sequence[Dict], branch: str) -> Tuple[List[Dict], Dict]:
    if branch in {"street", "fusion", "street_feature_control"}:
        filtered = [item for item in items if item["street_views"]]
    else:
        filtered = list(items)
    return filtered, {"branch": branch, "branch_input_records": len(items), "branch_retained_records": len(filtered), "branch_dropped_records": len(items) - len(filtered)}

def make_or_load_split(items: Sequence[Dict], split_path: Path, seed: int = 42, val_frac: float = 0.1) -> Tuple[List[Dict], List[Dict]]:
    ensure_dir(split_path.parent)
    if split_path.exists():
        split = json.load(open(split_path, "r", encoding="utf-8"))
        id_map = {item["id"]: item for item in items}
        return [id_map[i] for i in split["train_ids"] if i in id_map], [id_map[i] for i in split["val_ids"] if i in id_map]
    ids = [item["id"] for item in items]
    random.Random(seed).shuffle(ids)
    n_val = max(1, int(len(ids) * val_frac))
    val_ids, train_ids = ids[:n_val], ids[n_val:]
    save_json(split_path, {"seed": seed, "val_frac": val_frac, "train_ids": train_ids, "val_ids": val_ids})
    id_map = {item["id"]: item for item in items}
    return [id_map[i] for i in train_ids], [id_map[i] for i in val_ids]

def make_rgb_transform(image_size: int, train: bool) -> transforms.Compose:
    ops: List[object] = [transforms.Resize((image_size, image_size))]
    if train:
        ops += [transforms.RandomHorizontalFlip()]
    ops += [transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
    return transforms.Compose(ops)

class GlobalSatelliteDataset(Dataset):
    def __init__(self, items: Sequence[Dict], image_size: int = 224, train: bool = True):
        self.items = list(items)
        self.transform = make_rgb_transform(image_size, train)
    def __len__(self) -> int:
        return len(self.items)
    def __getitem__(self, idx: int):
        item = self.items[idx]
        image = Image.open(item["satellite"]).convert("RGB")
        return {"image": self.transform(image), "target": torch.tensor(item["reference"], dtype=torch.float32), "id": item["id"], "city": item["city"]}

class GlobalStreetViewDataset(Dataset):
    def __init__(self, items: Sequence[Dict], image_size: int = 224, train: bool = True, max_views: int = 10):
        self.items = [x for x in items if x["street_views"]]
        self.transform = make_rgb_transform(image_size, train)
        self.max_views = max_views
        self.image_size = image_size
    def __len__(self) -> int:
        return len(self.items)
    def _load_views(self, paths: Sequence[Path]) -> Tuple[torch.Tensor, torch.Tensor]:
        images, mask = [], []
        for path in list(paths)[: self.max_views]:
            image = Image.open(path).convert("RGB")
            images.append(self.transform(image))
            mask.append(1.0)
        while len(images) < self.max_views:
            images.append(torch.zeros_like(images[0]) if images else torch.zeros(3, self.image_size, self.image_size))
            mask.append(0.0)
        return torch.stack(images, dim=0), torch.tensor(mask, dtype=torch.float32)
    def __getitem__(self, idx: int):
        item = self.items[idx]
        views, mask = self._load_views(item["street_views"])
        return {"street_views": views, "street_mask": mask, "target": torch.tensor(item["reference"], dtype=torch.float32), "id": item["id"], "city": item["city"]}

class GlobalFusionDataset(Dataset):
    def __init__(self, items: Sequence[Dict], image_size: int = 224, train: bool = True, max_views: int = 10):
        self.items = [x for x in items if x["street_views"]]
        self.sat_transform = make_rgb_transform(image_size, train)
        self.stv_transform = make_rgb_transform(image_size, train)
        self.max_views = max_views
        self.image_size = image_size
    def __len__(self) -> int:
        return len(self.items)
    def __getitem__(self, idx: int):
        item = self.items[idx]
        sat = self.sat_transform(Image.open(item["satellite"]).convert("RGB"))
        street, mask = [], []
        for path in list(item["street_views"])[: self.max_views]:
            street.append(self.stv_transform(Image.open(path).convert("RGB")))
            mask.append(1.0)
        while len(street) < self.max_views:
            street.append(torch.zeros(3, self.image_size, self.image_size))
            mask.append(0.0)
        return {"image": sat, "street_views": torch.stack(street, dim=0), "street_mask": torch.tensor(mask, dtype=torch.float32), "target": torch.tensor(item["reference"], dtype=torch.float32), "id": item["id"], "city": item["city"]}
''',
    "evaluate/global_learned/models.py": r'''from __future__ import annotations

from typing import Dict, Optional

import timm
import torch
import torch.nn as nn

def reduce_backbone_output(out: torch.Tensor | dict | list | tuple) -> torch.Tensor:
    if isinstance(out, dict):
        for key in ["features", "x", "encoder_output", "last_hidden_state"]:
            if key in out:
                out = out[key]
                break
        else:
            out = next(iter(out.values()))
    if isinstance(out, (list, tuple)):
        out = out[-1]
    if out.dim() == 5:
        return out.mean(dim=(2, 3, 4))
    if out.dim() == 4:
        return out.mean(dim=(2, 3))
    if out.dim() == 3:
        return out.mean(dim=1)
    if out.dim() == 2:
        return out
    return out.view(out.size(0), -1)

class TimmEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.model_name = model_name
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feature_dim = getattr(self.backbone, "num_features", None) or getattr(self.backbone, "embed_dim", None)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        feat = reduce_backbone_output(out)
        if self.feature_dim is None:
            self.feature_dim = feat.size(-1)
        return feat

class PrithviRGBEncoder(nn.Module):
    def __init__(self, backbone_name: str = "prithvi_eo_v2_300", lora_r: int = 8):
        super().__init__()
        from terratorch.registry import BACKBONE_REGISTRY
        self.input_adapter = nn.Conv2d(3, 6, kernel_size=1, bias=False)
        self.backbone = BACKBONE_REGISTRY.build(backbone_name, pretrained=True, bands=["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"], num_frames=1)
        self.feature_dim: Optional[int] = None
        self.lora_applied = False
        if lora_r > 0:
            try:
                from peft import LoraConfig, get_peft_model
                cfg = LoraConfig(r=lora_r, lora_alpha=max(8, lora_r * 2), lora_dropout=0.05, bias="none", target_modules=["qkv", "q_proj", "k_proj", "v_proj", "proj"])
                self.backbone = get_peft_model(self.backbone, cfg)
                self.lora_applied = True
            except Exception:
                self.lora_applied = False
    def _forward_backbone(self, x: torch.Tensor):
        attempts = [x, x.unsqueeze(2), x.unsqueeze(1)]
        last_err = None
        for candidate in attempts:
            try:
                return self.backbone(candidate)
            except Exception as exc:
                last_err = exc
        raise RuntimeError(f"Prithvi forward failed for all input layouts. Last error: {last_err}")
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_adapter(x)
        out = self._forward_backbone(x)
        feat = reduce_backbone_output(out)
        if self.feature_dim is None:
            self.feature_dim = feat.size(-1)
        return feat

class AttentionPool(nn.Module):
    def __init__(self):
        super().__init__()
        self.score = nn.LazyLinear(1)
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        logits = self.score(x).squeeze(-1)
        if mask is not None:
            logits = logits.masked_fill(mask <= 0, -1e9)
        weights = torch.softmax(logits, dim=-1)
        return (x * weights.unsqueeze(-1)).sum(dim=1)

class StreetViewRegressor(nn.Module):
    def __init__(self, encoder: nn.Module, pooling: str = "mean", hidden_dim: int = 256):
        super().__init__()
        self.encoder = encoder
        self.pooling_name = pooling
        self.hidden_dim = hidden_dim
        self.pool = AttentionPool() if pooling == "attention" else None
        self.head = nn.Sequential(nn.LazyLinear(self.hidden_dim), nn.ReLU(), nn.Linear(self.hidden_dim, 1))
    def _pool_views(self, feats: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
        if self.pooling_name == "mean":
            if mask is None:
                return feats.mean(dim=1)
            denom = mask.sum(dim=1, keepdim=True).clamp_min(1.0)
            return (feats * mask.unsqueeze(-1)).sum(dim=1) / denom
        if self.pool is None:
            raise RuntimeError("Attention pooling requested but attention pool is not initialized")
        return self.pool(feats, mask)
    def forward_features(self, street_views: torch.Tensor, street_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        b, v, c, h, w = street_views.shape
        feats = self.encoder(street_views.view(b * v, c, h, w))
        feats = feats.view(b, v, -1)
        return self._pool_views(feats, street_mask)
    def forward(self, street_views: torch.Tensor, street_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        return self.head(self.forward_features(street_views, street_mask)).squeeze(-1)

class SatelliteRegressor(nn.Module):
    def __init__(self, encoder: nn.Module, hidden_dim: int = 256):
        super().__init__()
        self.encoder = encoder
        self.hidden_dim = hidden_dim
        self.head = nn.Sequential(nn.LazyLinear(self.hidden_dim), nn.ReLU(), nn.Linear(self.hidden_dim, 1))
    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.forward_features(x)).squeeze(-1)

class FusionRegressor(nn.Module):
    def __init__(self, sat_encoder: nn.Module, street_encoder: nn.Module, pooling: str = "mean", fusion_type: str = "late", hidden_dim: int = 256):
        super().__init__()
        self.sat_model = SatelliteRegressor(sat_encoder, hidden_dim=hidden_dim)
        self.street_model = StreetViewRegressor(street_encoder, pooling=pooling, hidden_dim=hidden_dim)
        self.fusion_type = fusion_type
        self.hidden_dim = hidden_dim
        self.gate = nn.LazyLinear(2) if fusion_type == "gated" else None
        self.head = nn.Sequential(nn.LazyLinear(self.hidden_dim), nn.ReLU(), nn.Linear(self.hidden_dim, 1))
    def forward_features(self, image: torch.Tensor, street_views: torch.Tensor, street_mask: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        sat_feat = self.sat_model.forward_features(image)
        stv_feat = self.street_model.forward_features(street_views, street_mask)
        if self.fusion_type == "gated":
            if self.gate is None:
                raise RuntimeError("Gated fusion requested but fusion gate is not initialized")
            fused = torch.cat([sat_feat, stv_feat], dim=-1)
            weights = torch.softmax(self.gate(fused), dim=-1)
            min_dim = min(sat_feat.size(-1), stv_feat.size(-1))
            combined = weights[:, :1] * sat_feat[:, :min_dim] + weights[:, 1:] * stv_feat[:, :min_dim]
        else:
            combined = torch.cat([sat_feat, stv_feat], dim=-1)
            weights = None
        return {"satellite": sat_feat, "street": stv_feat, "combined": combined, "weights": weights}
    def forward(self, image: torch.Tensor, street_views: torch.Tensor, street_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        return self.head(self.forward_features(image, street_views, street_mask)["combined"]).squeeze(-1)

def materialize_model(model: nn.Module, branch: str, batch: Dict[str, torch.Tensor], device: torch.device) -> None:
    was_training = model.training
    model.eval()
    with torch.no_grad():
        if branch == "satellite":
            _ = model(batch["image"].to(device))
        elif branch == "street":
            _ = model(batch["street_views"].to(device), batch["street_mask"].to(device))
        else:
            _ = model(batch["image"].to(device), batch["street_views"].to(device), batch["street_mask"].to(device))
    model.train(was_training)

def make_satellite_encoder(name: str, lora_r: int = 8) -> nn.Module:
    if name == "prithvi_rgb_lora":
        return PrithviRGBEncoder("prithvi_eo_v2_300", lora_r=lora_r)
    if name == "prithvi_rgb_lora_tl":
        return PrithviRGBEncoder("prithvi_eo_v2_300_tl", lora_r=lora_r)
    if name == "resnet50_sat":
        return TimmEncoder("resnet50")
    if name == "dinov2_sat":
        return TimmEncoder("vit_base_patch14_dinov2.lvd142m")
    raise ValueError(f"Unknown satellite model: {name}")

def make_street_encoder(name: str) -> nn.Module:
    if name == "resnet50":
        return TimmEncoder("resnet50")
    if name == "clip_vitb16":
        return TimmEncoder("vit_base_patch16_clip_224.openai")
    if name == "dinov2_vitb14":
        return TimmEncoder("vit_base_patch14_dinov2.lvd142m")
    if name == "swin_t":
        return TimmEncoder("swin_tiny_patch4_window7_224")
    raise ValueError(f"Unknown street model: {name}")
''',
    "evaluate/global_learned/train.py": r'''from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data import GlobalFusionDataset, GlobalSatelliteDataset, GlobalStreetViewDataset, filter_items_for_branch, load_global_items, make_or_load_split, resolve_data_root
from .models import FusionRegressor, SatelliteRegressor, StreetViewRegressor, make_satellite_encoder, make_street_encoder, materialize_model
from .utils import ExperimentConfig, GLOBAL_TASKS, append_experiment_log, compute_group_metrics, dump_config, ensure_dir, results_root, save_json, set_seed, write_rows_csv

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="CityLens learned global-task baselines")
    parser.add_argument("--task_name", default="all", choices=["all"] + GLOBAL_TASKS)
    parser.add_argument("--branch", default="satellite", choices=["satellite", "street", "fusion"])
    parser.add_argument("--satellite_model", default="prithvi_rgb_lora")
    parser.add_argument("--street_model", default="clip_vitb16")
    parser.add_argument("--fusion_type", default="late", choices=["late", "gated"])
    parser.add_argument("--pooling", default="mean", choices=["mean", "attention"])
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--weight_decay", type=float, default=1e-2)
    parser.add_argument("--val_frac", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--data_root", default=None)
    parser.add_argument("--skip_if_done", action="store_true")
    parser.add_argument("--resume", action="store_true")
    return parser.parse_args()

def build_dataloaders(branch: str, train_items: List[Dict], val_items: List[Dict], image_size: int, batch_size: int, num_workers: int) -> Tuple[DataLoader, DataLoader]:
    if branch == "satellite":
        train_ds, val_ds = GlobalSatelliteDataset(train_items, image_size=image_size, train=True), GlobalSatelliteDataset(val_items, image_size=image_size, train=False)
    elif branch == "street":
        train_ds, val_ds = GlobalStreetViewDataset(train_items, image_size=image_size, train=True), GlobalStreetViewDataset(val_items, image_size=image_size, train=False)
    else:
        train_ds, val_ds = GlobalFusionDataset(train_items, image_size=image_size, train=True), GlobalFusionDataset(val_items, image_size=image_size, train=False)
    return DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True), DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

def build_model(cfg: ExperimentConfig) -> nn.Module:
    if cfg.branch == "satellite":
        return SatelliteRegressor(make_satellite_encoder(cfg.satellite_model, lora_r=cfg.lora_r))
    if cfg.branch == "street":
        return StreetViewRegressor(make_street_encoder(cfg.street_model), pooling=cfg.pooling)
    return FusionRegressor(make_satellite_encoder(cfg.satellite_model, lora_r=cfg.lora_r), make_street_encoder(cfg.street_model), pooling=cfg.pooling, fusion_type=cfg.fusion_type)

def forward_batch(model: nn.Module, batch: Dict, branch: str, device: torch.device):
    target = batch["target"].to(device)
    if branch == "satellite":
        pred = model(batch["image"].to(device))
    elif branch == "street":
        pred = model(batch["street_views"].to(device), batch["street_mask"].to(device))
    else:
        pred = model(batch["image"].to(device), batch["street_views"].to(device), batch["street_mask"].to(device))
    return pred, target

def evaluate(model: nn.Module, loader: DataLoader, branch: str, device: torch.device):
    from .utils import compute_metrics
    model.eval()
    y_true, y_pred, predictions = [], [], []
    with torch.no_grad():
        for batch in loader:
            pred, target = forward_batch(model, batch, branch, device)
            pred_cpu, target_cpu = pred.detach().cpu().tolist(), target.detach().cpu().tolist()
            for sample_id, city, t, p in zip(batch["id"], batch["city"], target_cpu, pred_cpu):
                predictions.append({"id": sample_id, "city": city, "target": float(t), "prediction": float(p)})
            y_true.extend(target_cpu)
            y_pred.extend(pred_cpu)
    return compute_metrics(y_true, y_pred), predictions

def save_predictions(path: Path, rows: List[Dict]) -> None:
    write_rows_csv(path, rows, fieldnames=["id", "city", "target", "prediction"])

def save_per_city_metrics(path: Path, rows: List[Dict]) -> None:
    metrics = compute_group_metrics(rows, "city")
    save_json(path.with_suffix(".json"), metrics)
    write_rows_csv(path.with_suffix(".csv"), [{"city": city, **vals} for city, vals in metrics.items()], fieldnames=["city", "mse", "mae", "r2", "rmse"])

def run_task(task_name: str, args: argparse.Namespace, data_root: Path) -> None:
    cfg = ExperimentConfig(task_name=task_name, branch=args.branch, satellite_model=args.satellite_model, street_model=args.street_model, fusion_type=args.fusion_type if args.branch == "fusion" else "none", pooling=args.pooling if args.branch in {"street", "fusion"} else "none", image_size=args.image_size, batch_size=args.batch_size, epochs=args.epochs, lr=args.lr, weight_decay=args.weight_decay, val_frac=args.val_frac, seed=args.seed, lora_r=args.lora_r, num_workers=args.num_workers)
    task_root = results_root(data_root) / task_name / cfg.experiment_name()
    ckpt_dir = ensure_dir(task_root / "checkpoints")
    split_path = results_root(data_root) / "splits" / f"{task_name}_{cfg.branch}_seed{args.seed}_val{args.val_frac}.json"
    best_ckpt, last_ckpt = ckpt_dir / "best.pt", ckpt_dir / "last.pt"
    metrics_path, preds_path, per_city_path, history_path = task_root / "metrics.json", task_root / "val_predictions.csv", task_root / "per_city_metrics", task_root / "history.csv"
    if args.skip_if_done and best_ckpt.exists() and metrics_path.exists():
        metrics = json.load(open(metrics_path, "r", encoding="utf-8"))
        append_experiment_log(data_root, cfg, split_path, ckpt_dir, metrics, status="skipped")
        print(f"[skip] {task_name}: found existing checkpoint at {best_ckpt}")
        return
    items, load_report = load_global_items(task_name, data_root, return_report=True)
    filtered_items, filter_report = filter_items_for_branch(items, cfg.branch)
    if not filtered_items:
        raise RuntimeError(f"No records available for branch={cfg.branch} task={task_name}")
    train_items, val_items = make_or_load_split(filtered_items, split_path, seed=args.seed, val_frac=args.val_frac)
    save_json(task_root / "dataset_report.json", {**load_report, **filter_report, "train_records": len(train_items), "val_records": len(val_items)})
    train_loader, val_loader = build_dataloaders(cfg.branch, train_items, val_items, args.image_size, args.batch_size, args.num_workers)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(cfg).to(device)
    sample_batch = next(iter(train_loader if len(train_loader) > 0 else val_loader))
    materialize_model(model, cfg.branch, sample_batch, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    loss_fn, start_epoch, best_rmse, history_rows = nn.MSELoss(), 1, float("inf"), []
    if args.resume and last_ckpt.exists():
        state = torch.load(last_ckpt, map_location=device)
        model.load_state_dict(state["model"], strict=True)
        optimizer.load_state_dict(state["optimizer"])
        start_epoch = int(state["epoch"]) + 1
        best_rmse = float(state.get("best_rmse", best_rmse))
        print(f"[resume] {task_name}: resuming from epoch {start_epoch}")
    dump_config(task_root / "config.json", cfg)
    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        running, seen = 0.0, 0
        prog = tqdm(train_loader, desc=f"{task_name} epoch {epoch}/{args.epochs}", leave=False)
        for batch in prog:
            pred, target = forward_batch(model, batch, cfg.branch, device)
            loss = loss_fn(pred, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            batch_n = target.numel()
            running += loss.item() * batch_n
            seen += batch_n
            prog.set_postfix(train_loss=running / max(1, seen))
        val_metrics, predictions = evaluate(model, val_loader, cfg.branch, device)
        history_rows.append({"epoch": epoch, "train_loss": running / max(1, seen), **val_metrics})
        if val_metrics["rmse"] < best_rmse:
            best_rmse = val_metrics["rmse"]
            torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "best_rmse": best_rmse, "config": cfg.__dict__}, best_ckpt)
            save_predictions(preds_path, predictions)
            save_per_city_metrics(per_city_path, predictions)
            save_json(metrics_path, {"best_epoch": epoch, "val_records": len(predictions), **val_metrics})
        torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "best_rmse": best_rmse, "config": cfg.__dict__}, last_ckpt)
        with open(history_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "mse", "mae", "r2", "rmse"])
            writer.writeheader()
            writer.writerows(history_rows)
        print(f"[{task_name}] epoch={epoch} val_rmse={val_metrics['rmse']:.4f} val_r2={val_metrics['r2']:.4f}")
    final_metrics = json.load(open(metrics_path, "r", encoding="utf-8")) if metrics_path.exists() else {}
    append_experiment_log(data_root, cfg, split_path, ckpt_dir, final_metrics, status="finished")

def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    data_root = resolve_data_root(args.data_root)
    tasks = GLOBAL_TASKS if args.task_name == "all" else [args.task_name]
    for task_name in tasks:
        run_task(task_name, args, data_root)

if __name__ == "__main__":
    main()
''',
    "evaluate/global_learned/feature_control.py": r'''from __future__ import annotations

import argparse
import json
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data import GlobalStreetViewDataset, filter_items_for_branch, load_global_items, make_or_load_split, resolve_data_root
from .models import StreetViewRegressor, make_street_encoder
from .utils import ExperimentConfig, GLOBAL_TASKS, append_experiment_log, compute_group_metrics, compute_metrics, dump_config, ensure_dir, results_root, save_json, set_seed, write_rows_csv

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="CityLens low-cost street-view control")
    parser.add_argument("--task_name", default="all", choices=["all"] + GLOBAL_TASKS)
    parser.add_argument("--street_model", default="resnet50")
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--val_frac", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--data_root", default=None)
    parser.add_argument("--skip_if_done", action="store_true")
    return parser.parse_args()

def make_loader(items: List[Dict], image_size: int, batch_size: int, num_workers: int) -> DataLoader:
    ds = GlobalStreetViewDataset(items, image_size=image_size, train=False)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

def extract_embeddings(loader: DataLoader, model: StreetViewRegressor, device: torch.device) -> Tuple[np.ndarray, np.ndarray, List[str], List[str]]:
    feats, targets, ids, cities = [], [], [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, leave=False, desc="extract"):
            emb = model.forward_features(batch["street_views"].to(device), batch["street_mask"].to(device)).detach().cpu().numpy()
            feats.append(emb)
            targets.extend(batch["target"].cpu().numpy().tolist())
            ids.extend(list(batch["id"]))
            cities.extend(list(batch["city"]))
    return np.concatenate(feats, axis=0), np.asarray(targets, dtype=np.float32), ids, cities

def save_predictions(path: Path, rows: List[Dict]) -> None:
    write_rows_csv(path, rows, fieldnames=["id", "city", "target", "prediction"])

def run_task(task_name: str, args: argparse.Namespace, data_root: Path) -> None:
    cfg = ExperimentConfig(task_name=task_name, branch="street_feature_control", satellite_model="none", street_model=args.street_model, fusion_type="none", pooling="mean", image_size=args.image_size, batch_size=args.batch_size, epochs=0, lr=0.0, weight_decay=0.0, val_frac=args.val_frac, seed=args.seed, lora_r=0, num_workers=args.num_workers)
    task_root = results_root(data_root) / task_name / cfg.experiment_name()
    ckpt_dir = ensure_dir(task_root / "checkpoints")
    metrics_path, preds_path, per_city_path = task_root / "metrics.json", task_root / "val_predictions.csv", task_root / "per_city_metrics"
    split_path = results_root(data_root) / "splits" / f"{task_name}_{cfg.branch}_seed{args.seed}_val{args.val_frac}.json"
    artifact_path = ckpt_dir / "lasso.pkl"
    if args.skip_if_done and metrics_path.exists() and artifact_path.exists():
        metrics = json.load(open(metrics_path, "r", encoding="utf-8"))
        append_experiment_log(data_root, cfg, split_path, ckpt_dir, metrics, status="skipped")
        print(f"[skip] {task_name}: feature control already exists")
        return
    items, load_report = load_global_items(task_name, data_root, return_report=True)
    filtered_items, filter_report = filter_items_for_branch(items, cfg.branch)
    train_items, val_items = make_or_load_split(filtered_items, split_path, seed=args.seed, val_frac=args.val_frac)
    save_json(task_root / "dataset_report.json", {**load_report, **filter_report, "train_records": len(train_items), "val_records": len(val_items)})
    train_loader = make_loader(train_items, args.image_size, args.batch_size, args.num_workers)
    val_loader = make_loader(val_items, args.image_size, args.batch_size, args.num_workers)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = StreetViewRegressor(make_street_encoder(args.street_model), pooling="mean").to(device)
    x_train, y_train, _, _ = extract_embeddings(train_loader, model, device)
    x_val, y_val, val_ids, val_cities = extract_embeddings(val_loader, model, device)
    scaler = StandardScaler()
    x_train_scaled, x_val_scaled = scaler.fit_transform(x_train), scaler.transform(x_val)
    reg = LassoCV(cv=5, random_state=args.seed, max_iter=10000)
    reg.fit(x_train_scaled, y_train)
    y_pred = reg.predict(x_val_scaled)
    metrics = compute_metrics(y_val, y_pred)
    prediction_rows = [{"id": sample_id, "city": city, "target": float(target), "prediction": float(pred)} for sample_id, city, target, pred in zip(val_ids, val_cities, y_val.tolist(), y_pred.tolist())]
    dump_config(task_root / "config.json", cfg)
    save_json(metrics_path, {"val_records": len(prediction_rows), **metrics})
    save_predictions(preds_path, prediction_rows)
    city_metrics = compute_group_metrics(prediction_rows, "city")
    save_json(per_city_path.with_suffix(".json"), city_metrics)
    write_rows_csv(per_city_path.with_suffix(".csv"), [{"city": city, **vals} for city, vals in city_metrics.items()], fieldnames=["city", "mse", "mae", "r2", "rmse"])
    with open(artifact_path, "wb") as f:
        pickle.dump({"scaler": scaler, "regressor": reg}, f)
    append_experiment_log(data_root, cfg, split_path, ckpt_dir, metrics, status="finished")
    print(f"[{task_name}] feature-control rmse={metrics['rmse']:.4f} r2={metrics['r2']:.4f}")

def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    data_root = resolve_data_root(args.data_root)
    tasks = GLOBAL_TASKS if args.task_name == "all" else [args.task_name]
    for task_name in tasks:
        run_task(task_name, args, data_root)

if __name__ == "__main__":
    main()
''',
    "evaluate/global_learned/explain.py": r'''from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
from captum.attr import IntegratedGradients
from PIL import Image

from .data import GlobalFusionDataset, GlobalSatelliteDataset, GlobalStreetViewDataset, filter_items_for_branch, load_global_items, make_or_load_split, resolve_data_root
from .models import FusionRegressor, SatelliteRegressor, StreetViewRegressor, make_satellite_encoder, make_street_encoder, materialize_model
from .utils import GLOBAL_TASKS, ensure_dir, results_root, save_json, set_seed, write_rows_csv

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Explain learned CityLens global-task models")
    parser.add_argument("--task_name", required=True, choices=GLOBAL_TASKS)
    parser.add_argument("--branch", required=True, choices=["satellite", "street", "fusion"])
    parser.add_argument("--checkpoint_dir", required=True)
    parser.add_argument("--satellite_model", default="prithvi_rgb_lora")
    parser.add_argument("--street_model", default="clip_vitb16")
    parser.add_argument("--fusion_type", default="late", choices=["late", "gated"])
    parser.add_argument("--pooling", default="mean", choices=["mean", "attention"])
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--data_root", default=None)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--val_frac", type=float, default=0.1)
    parser.add_argument("--max_samples", type=int, default=8)
    return parser.parse_args()

def apply_checkpoint_config(args: argparse.Namespace) -> argparse.Namespace:
    config_path = Path(args.checkpoint_dir) / "config.json"
    if config_path.exists():
        cfg = json.load(open(config_path, "r", encoding="utf-8"))
        for key in ["branch", "satellite_model", "street_model", "fusion_type", "pooling", "image_size", "seed", "val_frac"]:
            if key in cfg:
                setattr(args, key, cfg[key])
    return args

def build_model(args: argparse.Namespace) -> torch.nn.Module:
    if args.branch == "satellite":
        return SatelliteRegressor(make_satellite_encoder(args.satellite_model))
    if args.branch == "street":
        return StreetViewRegressor(make_street_encoder(args.street_model), pooling=args.pooling)
    return FusionRegressor(make_satellite_encoder(args.satellite_model), make_street_encoder(args.street_model), pooling=args.pooling, fusion_type=args.fusion_type)

def load_val_samples(args: argparse.Namespace, data_root: Path) -> List[Dict]:
    items, _ = load_global_items(args.task_name, data_root, return_report=True)
    filtered_items, _ = filter_items_for_branch(items, args.branch)
    split_path = results_root(data_root) / "splits" / f"{args.task_name}_{args.branch}_seed{args.seed}_val{args.val_frac}.json"
    _, val_items = make_or_load_split(filtered_items, split_path, seed=args.seed, val_frac=args.val_frac)
    if args.branch == "satellite":
        ds = GlobalSatelliteDataset(val_items, image_size=args.image_size, train=False)
    elif args.branch == "street":
        ds = GlobalStreetViewDataset(val_items, image_size=args.image_size, train=False)
    else:
        ds = GlobalFusionDataset(val_items, image_size=args.image_size, train=False)
    return [ds[i] for i in range(min(len(ds), args.max_samples))]

def save_heatmap_png(attr: np.ndarray, path: Path) -> None:
    arr = np.abs(attr).mean(axis=0)
    arr = arr - arr.min()
    arr = arr / max(arr.max(), 1e-8)
    Image.fromarray((arr * 255).astype(np.uint8)).save(path)

def save_input_preview(image_tensor: torch.Tensor, path: Path) -> None:
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = (image_tensor.detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    Image.fromarray((image * 255).astype(np.uint8)).save(path)

def satellite_integrated_gradients(model: torch.nn.Module, sample: Dict, device: torch.device) -> np.ndarray:
    model.eval()
    x = sample["image"].unsqueeze(0).to(device)
    ig = IntegratedGradients(model)
    return ig.attribute(x, target=None).detach().cpu().numpy()[0]

def street_leave_one_view_out(model: torch.nn.Module, sample: Dict, device: torch.device) -> List[Dict]:
    model.eval()
    x, mask = sample["street_views"].unsqueeze(0).to(device), sample["street_mask"].unsqueeze(0).to(device)
    with torch.no_grad():
        base = model(x, mask).item()
    rows = []
    for i in range(x.size(1)):
        if mask[0, i].item() <= 0:
            continue
        x_mod, mask_mod = x.clone(), mask.clone()
        x_mod[0, i] = 0
        mask_mod[0, i] = 0
        with torch.no_grad():
            pred = model(x_mod, mask_mod).item()
        rows.append({"view_index": i, "base_prediction": base, "ablated_prediction": pred, "delta": base - pred})
    return rows

def fusion_modality_ablation(model: torch.nn.Module, sample: Dict, device: torch.device) -> Dict:
    model.eval()
    image = sample["image"].unsqueeze(0).to(device)
    views = sample["street_views"].unsqueeze(0).to(device)
    mask = sample["street_mask"].unsqueeze(0).to(device)
    with torch.no_grad():
        full = model(image, views, mask).item()
        sat_zero = model(torch.zeros_like(image), views, mask).item()
        stv_zero = model(image, torch.zeros_like(views), torch.zeros_like(mask)).item()
    return {"id": sample["id"], "city": sample["city"], "full_prediction": full, "no_satellite_prediction": sat_zero, "no_street_prediction": stv_zero, "satellite_contribution_delta": full - sat_zero, "street_contribution_delta": full - stv_zero}

def main() -> None:
    args = apply_checkpoint_config(parse_args())
    set_seed(args.seed)
    data_root = resolve_data_root(args.data_root)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(args).to(device)
    samples = load_val_samples(args, data_root)
    if not samples:
        raise RuntimeError("No validation samples available for explainability")
    materialize_model(model, args.branch, samples[0], device)
    ckpt = torch.load(Path(args.checkpoint_dir) / "checkpoints" / "best.pt", map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    out_dir = ensure_dir(Path(args.checkpoint_dir) / "explain")
    save_json(out_dir / "explainability_manifest.json", {"branch": args.branch, "task_name": args.task_name, "max_samples": len(samples), "feasibility_note": "Integrated gradients, leave-one-view-out, and modality ablation are feasible here. True multispectral spectral ablation is not supported because CityLens is RGB-only."})
    if args.branch == "satellite":
        rows = []
        for idx, sample in enumerate(samples):
            attr = satellite_integrated_gradients(model, sample, device)
            sample_dir = ensure_dir(out_dir / f"sample_{idx:02d}_{sample['id']}")
            np.save(sample_dir / "integrated_gradients.npy", attr)
            save_input_preview(sample["image"], sample_dir / "input.png")
            save_heatmap_png(attr, sample_dir / "heatmap.png")
            rows.append({"id": sample["id"], "city": sample["city"], "shape": list(attr.shape), "mean_abs": float(np.abs(attr).mean()), "max_abs": float(np.abs(attr).max())})
        save_json(out_dir / "integrated_gradients_summary.json", {"rows": rows})
    elif args.branch == "street":
        rows = []
        for sample in samples:
            for row in street_leave_one_view_out(model, sample, device):
                rows.append({"id": sample["id"], "city": sample["city"], **row})
        save_json(out_dir / "leave_one_view_out.json", {"rows": rows})
        write_rows_csv(out_dir / "leave_one_view_out.csv", rows, fieldnames=["id", "city", "view_index", "base_prediction", "ablated_prediction", "delta"])
    else:
        rows = [fusion_modality_ablation(model, sample, device) for sample in samples]
        save_json(out_dir / "modality_ablation.json", {"rows": rows})
        write_rows_csv(out_dir / "modality_ablation.csv", rows, fieldnames=["id", "city", "full_prediction", "no_satellite_prediction", "no_street_prediction", "satellite_contribution_delta", "street_contribution_delta"])
    print(f"Saved explainability outputs to {out_dir}")

if __name__ == "__main__":
    main()
''',
}

for rel_path, text in UPDATED_FILES.items():
    out_path = REPO_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(text.rstrip() + "\n", encoding="utf-8")
    print("Updated", out_path)

print("Refreshed bootstrapped learned-model files to audited versions.")

Updated /content/CityLens/evaluate/global_learned/utils.py
Updated /content/CityLens/evaluate/global_learned/data.py
Updated /content/CityLens/evaluate/global_learned/models.py
Updated /content/CityLens/evaluate/global_learned/train.py
Updated /content/CityLens/evaluate/global_learned/feature_control.py
Updated /content/CityLens/evaluate/global_learned/explain.py
Refreshed bootstrapped learned-model files to audited versions.


In [52]:
# Refresh train/utils for target-scaling and optimizer-group improvements
from pathlib import Path

REPO_ROOT = Path("/content/CityLens")
UPDATED_TRAINING_FILES = {
    "evaluate/global_learned/utils.py": r'''import csv
import json
import random
from collections import defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Sequence

import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

GLOBAL_TASKS = ["gdp", "pop", "acc2health", "carbon", "build_height"]

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def slugify(text: str) -> str:
    return text.replace("/", "_").replace(" ", "_").replace(":", "_").replace(",", "_").replace("__", "_").strip("_")

def results_root(data_root: Path) -> Path:
    return data_root / "Results" / "global_learned"

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

@dataclass
class ExperimentConfig:
    task_name: str
    branch: str
    satellite_model: str
    street_model: str
    fusion_type: str
    pooling: str
    image_size: int
    batch_size: int
    epochs: int
    lr: float
    weight_decay: float
    backbone_lr: float
    head_lr: float
    val_frac: float
    seed: int
    lora_r: int
    target_transform: str
    num_workers: int

    def experiment_name(self) -> str:
        bits = [self.task_name, self.branch, self.satellite_model, self.street_model, self.fusion_type, self.pooling, f"bs{self.batch_size}", f"ep{self.epochs}", f"lr{self.lr}", f"blr{self.backbone_lr}", f"hlr{self.head_lr}", f"tt{self.target_transform}", f"seed{self.seed}"]
        return slugify("-".join([b for b in bits if b and b != "none"]))

def compute_metrics(y_true: Iterable[float], y_pred: Iterable[float]) -> Dict[str, float]:
    y_true = np.asarray(list(y_true), dtype=np.float32)
    y_pred = np.asarray(list(y_pred), dtype=np.float32)
    if len(y_true) == 0:
        return {"mse": float("nan"), "mae": float("nan"), "r2": float("nan"), "rmse": float("nan")}
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    return {"mse": float(mse), "mae": float(mae), "r2": float(r2), "rmse": float(rmse)}

def save_json(path: Path, payload: Dict) -> None:
    ensure_dir(path.parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

def append_csv(path: Path, row: Dict, fieldnames: List[str]) -> None:
    ensure_dir(path.parent)
    write_header = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

def write_rows_csv(path: Path, rows: Sequence[Dict], fieldnames: List[str]) -> None:
    ensure_dir(path.parent)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def append_experiment_log(data_root: Path, config: ExperimentConfig, split_path: Path, checkpoint_dir: Path, metrics: Dict[str, float], status: str) -> None:
    row = {"timestamp": datetime.utcnow().isoformat(timespec="seconds"), "task_name": config.task_name, "branch": config.branch, "satellite_model": config.satellite_model, "street_model": config.street_model, "fusion_type": config.fusion_type, "pooling": config.pooling, "image_size": config.image_size, "batch_size": config.batch_size, "epochs": config.epochs, "lr": config.lr, "weight_decay": config.weight_decay, "backbone_lr": config.backbone_lr, "head_lr": config.head_lr, "val_frac": config.val_frac, "seed": config.seed, "lora_r": config.lora_r, "target_transform": config.target_transform, "split_path": str(split_path), "checkpoint_dir": str(checkpoint_dir), "status": status, **metrics}
    append_csv(results_root(data_root) / "experiment_log.csv", row, fieldnames=list(row.keys()))

def dump_config(path: Path, config: ExperimentConfig) -> None:
    save_json(path, asdict(config))

def compute_group_metrics(rows: Sequence[Dict], group_key: str) -> Dict[str, Dict[str, float]]:
    grouped = defaultdict(lambda: {"target": [], "prediction": []})
    for row in rows:
        group = row.get(group_key)
        if group in (None, ""):
            continue
        grouped[str(group)]["target"].append(float(row["target"]))
        grouped[str(group)]["prediction"].append(float(row["prediction"]))
    return {group: compute_metrics(vals["target"], vals["prediction"]) for group, vals in grouped.items()}
''',
    "evaluate/global_learned/train.py": r'''from __future__ import annotations

import argparse
import csv
import json
import math
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from .data import GlobalFusionDataset, GlobalSatelliteDataset, GlobalStreetViewDataset, filter_items_for_branch, load_global_items, make_or_load_split, resolve_data_root
from .models import FusionRegressor, SatelliteRegressor, StreetViewRegressor, make_satellite_encoder, make_street_encoder, materialize_model
from .utils import ExperimentConfig, GLOBAL_TASKS, append_experiment_log, compute_group_metrics, compute_metrics, dump_config, ensure_dir, results_root, save_json, set_seed, write_rows_csv

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="CityLens learned global-task baselines")
    parser.add_argument("--task_name", default="all", choices=["all"] + GLOBAL_TASKS)
    parser.add_argument("--branch", default="satellite", choices=["satellite", "street", "fusion"])
    parser.add_argument("--satellite_model", default="prithvi_rgb_lora")
    parser.add_argument("--street_model", default="clip_vitb16")
    parser.add_argument("--fusion_type", default="late", choices=["late", "gated"])
    parser.add_argument("--pooling", default="mean", choices=["mean", "attention"])
    parser.add_argument("--image_size", type=int, default=224)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--backbone_lr", type=float, default=None)
    parser.add_argument("--head_lr", type=float, default=1e-3)
    parser.add_argument("--weight_decay", type=float, default=1e-2)
    parser.add_argument("--val_frac", type=float, default=0.1)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--lora_r", type=int, default=8)
    parser.add_argument("--target_transform", default="auto", choices=["auto", "raw", "log1p"])
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--data_root", default=None)
    parser.add_argument("--skip_if_done", action="store_true")
    parser.add_argument("--resume", action="store_true")
    return parser.parse_args()

class TargetTransform:
    def __init__(self, mode: str):
        self.mode = mode
    def encode_tensor(self, x: torch.Tensor) -> torch.Tensor:
        if self.mode == "log1p":
            return torch.log1p(torch.clamp(x, min=0.0))
        return x
    def decode_tensor(self, x: torch.Tensor) -> torch.Tensor:
        if self.mode == "log1p":
            return torch.expm1(x)
        return x
    def decode_float(self, value: float) -> float:
        if self.mode == "log1p":
            return max(0.0, math.expm1(value))
        return value

def resolve_target_transform(task_name: str, requested: str) -> str:
    if requested == "auto":
        return "log1p" if task_name in GLOBAL_TASKS else "raw"
    return requested

def build_dataloaders(branch: str, train_items: List[Dict], val_items: List[Dict], image_size: int, batch_size: int, num_workers: int) -> Tuple[DataLoader, DataLoader]:
    if branch == "satellite":
        train_ds, val_ds = GlobalSatelliteDataset(train_items, image_size=image_size, train=True), GlobalSatelliteDataset(val_items, image_size=image_size, train=False)
    elif branch == "street":
        train_ds, val_ds = GlobalStreetViewDataset(train_items, image_size=image_size, train=True), GlobalStreetViewDataset(val_items, image_size=image_size, train=False)
    else:
        train_ds, val_ds = GlobalFusionDataset(train_items, image_size=image_size, train=True), GlobalFusionDataset(val_items, image_size=image_size, train=False)
    return DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True), DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

def build_model(cfg: ExperimentConfig) -> nn.Module:
    if cfg.branch == "satellite":
        return SatelliteRegressor(make_satellite_encoder(cfg.satellite_model, lora_r=cfg.lora_r))
    if cfg.branch == "street":
        return StreetViewRegressor(make_street_encoder(cfg.street_model), pooling=cfg.pooling)
    return FusionRegressor(make_satellite_encoder(cfg.satellite_model, lora_r=cfg.lora_r), make_street_encoder(cfg.street_model), pooling=cfg.pooling, fusion_type=cfg.fusion_type)

def forward_batch(model: nn.Module, batch: Dict, branch: str, device: torch.device, target_transform: TargetTransform):
    raw_target = batch["target"].to(device)
    target = target_transform.encode_tensor(raw_target)
    if branch == "satellite":
        pred = model(batch["image"].to(device))
    elif branch == "street":
        pred = model(batch["street_views"].to(device), batch["street_mask"].to(device))
    else:
        pred = model(batch["image"].to(device), batch["street_views"].to(device), batch["street_mask"].to(device))
    return pred, target, raw_target

def evaluate(model: nn.Module, loader: DataLoader, branch: str, device: torch.device, target_transform: TargetTransform):
    model.eval()
    y_true, y_pred, predictions = [], [], []
    with torch.no_grad():
        for batch in loader:
            pred, _, raw_target = forward_batch(model, batch, branch, device, target_transform)
            pred_raw = target_transform.decode_tensor(pred)
            pred_cpu, target_cpu = pred_raw.detach().cpu().tolist(), raw_target.detach().cpu().tolist()
            for sample_id, city, t, p in zip(batch["id"], batch["city"], target_cpu, pred_cpu):
                predictions.append({"id": sample_id, "city": city, "target": float(t), "prediction": float(p)})
            y_true.extend(target_cpu)
            y_pred.extend(pred_cpu)
    return compute_metrics(y_true, y_pred), predictions

def build_optimizer(model: nn.Module, cfg: ExperimentConfig) -> torch.optim.Optimizer:
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(key in name for key in ["head", "input_adapter", "gate", "pool"]):
            head_params.append(param)
        else:
            backbone_params.append(param)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": cfg.backbone_lr, "weight_decay": cfg.weight_decay})
    if head_params:
        groups.append({"params": head_params, "lr": cfg.head_lr, "weight_decay": cfg.weight_decay})
    return torch.optim.AdamW(groups)

def save_predictions(path: Path, rows: List[Dict]) -> None:
    write_rows_csv(path, rows, fieldnames=["id", "city", "target", "prediction"])

def save_per_city_metrics(path: Path, rows: List[Dict]) -> None:
    metrics = compute_group_metrics(rows, "city")
    save_json(path.with_suffix(".json"), metrics)
    write_rows_csv(path.with_suffix(".csv"), [{"city": city, **vals} for city, vals in metrics.items()], fieldnames=["city", "mse", "mae", "r2", "rmse"])

def run_task(task_name: str, args: argparse.Namespace, data_root: Path) -> None:
    cfg = ExperimentConfig(task_name=task_name, branch=args.branch, satellite_model=args.satellite_model, street_model=args.street_model, fusion_type=args.fusion_type if args.branch == "fusion" else "none", pooling=args.pooling if args.branch in {"street", "fusion"} else "none", image_size=args.image_size, batch_size=args.batch_size, epochs=args.epochs, lr=args.lr, weight_decay=args.weight_decay, backbone_lr=args.backbone_lr if args.backbone_lr is not None else args.lr, head_lr=args.head_lr, val_frac=args.val_frac, seed=args.seed, lora_r=args.lora_r, target_transform=resolve_target_transform(task_name, args.target_transform), num_workers=args.num_workers)
    task_root = results_root(data_root) / task_name / cfg.experiment_name()
    ckpt_dir = ensure_dir(task_root / "checkpoints")
    split_path = results_root(data_root) / "splits" / f"{task_name}_{cfg.branch}_seed{args.seed}_val{args.val_frac}.json"
    best_ckpt, last_ckpt = ckpt_dir / "best.pt", ckpt_dir / "last.pt"
    metrics_path, preds_path, per_city_path, history_path = task_root / "metrics.json", task_root / "val_predictions.csv", task_root / "per_city_metrics", task_root / "history.csv"
    if args.skip_if_done and best_ckpt.exists() and metrics_path.exists():
        metrics = json.load(open(metrics_path, "r", encoding="utf-8"))
        print(f"[skip] {task_name}: found existing checkpoint at {best_ckpt}")
        append_experiment_log(data_root, cfg, split_path, ckpt_dir, metrics, status="skipped")
        return
    items, load_report = load_global_items(task_name, data_root, return_report=True)
    filtered_items, filter_report = filter_items_for_branch(items, cfg.branch)
    if not filtered_items:
        raise RuntimeError(f"No records available for branch={cfg.branch} task={task_name}")
    train_items, val_items = make_or_load_split(filtered_items, split_path, seed=args.seed, val_frac=args.val_frac)
    save_json(task_root / "dataset_report.json", {**load_report, **filter_report, "train_records": len(train_items), "val_records": len(val_items)})
    train_loader, val_loader = build_dataloaders(cfg.branch, train_items, val_items, args.image_size, args.batch_size, args.num_workers)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(cfg).to(device)
    sample_batch = next(iter(train_loader if len(train_loader) > 0 else val_loader))
    materialize_model(model, cfg.branch, sample_batch, device)
    optimizer = build_optimizer(model, cfg)
    loss_fn, target_transform = nn.MSELoss(), TargetTransform(cfg.target_transform)
    start_epoch, best_rmse, history_rows = 1, float("inf"), []
    if args.resume and last_ckpt.exists():
        state = torch.load(last_ckpt, map_location=device)
        model.load_state_dict(state["model"], strict=True)
        try:
            optimizer.load_state_dict(state["optimizer"])
        except ValueError:
            print(f"[resume] {task_name}: optimizer state mismatch, continuing with fresh optimizer")
        start_epoch = int(state["epoch"]) + 1
        best_rmse = float(state.get("best_rmse", best_rmse))
        print(f"[resume] {task_name}: resuming from epoch {start_epoch}")
    dump_config(task_root / "config.json", cfg)
    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        running, seen = 0.0, 0
        prog = tqdm(train_loader, desc=f"{task_name} epoch {epoch}/{args.epochs}", leave=False)
        for batch in prog:
            pred, target, _ = forward_batch(model, batch, cfg.branch, device, target_transform)
            loss = loss_fn(pred, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            batch_n = target.numel()
            running += loss.item() * batch_n
            seen += batch_n
            prog.set_postfix(train_loss=running / max(1, seen))
        val_metrics, predictions = evaluate(model, val_loader, cfg.branch, device, target_transform)
        pred_values, target_values = [row["prediction"] for row in predictions], [row["target"] for row in predictions]
        row = {"epoch": epoch, "train_loss": running / max(1, seen), "pred_mean": float(sum(pred_values) / max(1, len(pred_values))), "pred_std": float(torch.tensor(pred_values).std(unbiased=False).item()) if pred_values else float("nan"), "target_mean": float(sum(target_values) / max(1, len(target_values))), "target_std": float(torch.tensor(target_values).std(unbiased=False).item()) if target_values else float("nan"), **val_metrics}
        history_rows.append(row)
        if val_metrics["rmse"] < best_rmse:
            best_rmse = val_metrics["rmse"]
            torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "best_rmse": best_rmse, "config": cfg.__dict__}, best_ckpt)
            save_predictions(preds_path, predictions)
            save_per_city_metrics(per_city_path, predictions)
            save_json(metrics_path, {"best_epoch": epoch, "val_records": len(predictions), **val_metrics})
        torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "best_rmse": best_rmse, "config": cfg.__dict__}, last_ckpt)
        with open(history_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "pred_mean", "pred_std", "target_mean", "target_std", "mse", "mae", "r2", "rmse"])
            writer.writeheader()
            writer.writerows(history_rows)
        print(f"[{task_name}] epoch={epoch} val_rmse={val_metrics['rmse']:.4f} val_r2={val_metrics['r2']:.4f} pred_std={row['pred_std']:.4f} target_std={row['target_std']:.4f} transform={cfg.target_transform}")
    final_metrics = json.load(open(metrics_path, "r", encoding="utf-8")) if metrics_path.exists() else {}
    append_experiment_log(data_root, cfg, split_path, ckpt_dir, final_metrics, status="finished")

def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    data_root = resolve_data_root(args.data_root)
    tasks = GLOBAL_TASKS if args.task_name == "all" else [args.task_name]
    for task_name in tasks:
        run_task(task_name, args, data_root)

if __name__ == "__main__":
    main()
''',
}

for rel_path, text in UPDATED_TRAINING_FILES.items():
    out_path = REPO_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(text.rstrip() + "\n", encoding="utf-8")
    print("Updated", out_path)

print("Refreshed train/utils with target scaling and optimizer-group improvements.")

Updated /content/CityLens/evaluate/global_learned/utils.py
Updated /content/CityLens/evaluate/global_learned/train.py
Refreshed train/utils with target scaling and optimizer-group improvements.


In [53]:
'''
# Learned-model orchestrator for all 5 global tasks
import os
import subprocess
from pathlib import Path

DATA_ROOT = os.environ.get("CITYLENS_DATA_ROOT")
if not DATA_ROOT:
    raise SystemExit("CITYLENS_DATA_ROOT not set. Run section 3 first.")
print("CITYLENS_DATA_ROOT =", DATA_ROOT)

# Safe defaults for Run-all in Colab:
# - low-cost control ON
# - satellite branch ON (main experiment)
# - street/fusion/explainability OFF by default because they are much longer
RUN_FEATURE_CONTROL = False
RUN_SATELLITE = True
RUN_STREET = False
RUN_FUSION = False
RUN_EXPLAIN = False

# Stage 0: low-cost control
FEATURE_CONTROL_MODEL = "resnet50"

# Stage 1: satellite-only
SATELLITE_MODELS = [
    "prithvi_rgb_lora",  # main adapted Prithvi baseline
    "dinov2_sat",        # RGB satellite fallback
]

# Stage 2: street-only
STREET_MODELS = [
    "resnet50",
    "clip_vitb16",
    "dinov2_vitb14",
]
INCLUDE_SWIN_T = False
if INCLUDE_SWIN_T:
    STREET_MODELS.append("swin_t")

# Stage 3: fusion
FUSION_SAT_MODEL = "prithvi_rgb_lora"
FUSION_STREET_MODEL = "clip_vitb16"
FUSION_TYPES = ["late", "gated"]

EPOCHS = 5
BATCH_SIZE = 8
IMAGE_SIZE = 224
SEED = 42
NUM_WORKERS = 2


def run(cmd):
    print("\n$", " ".join(cmd))
    r = subprocess.run(cmd, cwd="/content/CityLens", text=True)
    if r.returncode != 0:
        raise SystemExit(f"Command failed: {' '.join(cmd)}")

base_cmd = [
    "python", "-m", "evaluate.global_learned.train",
    "--task_name", "all",
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--image_size", str(IMAGE_SIZE),
    "--seed", str(SEED),
    "--num_workers", str(NUM_WORKERS),
    "--skip_if_done",
    "--resume",
]

if RUN_FEATURE_CONTROL:
    run([
        "python", "-m", "evaluate.global_learned.feature_control",
        "--task_name", "all",
        "--street_model", FEATURE_CONTROL_MODEL,
        "--image_size", str(IMAGE_SIZE),
        "--batch_size", str(BATCH_SIZE),
        "--seed", str(SEED),
        "--num_workers", str(NUM_WORKERS),
        "--skip_if_done",
    ])

if RUN_SATELLITE:
    for sat_model in SATELLITE_MODELS:
        run(base_cmd + [
            "--branch", "satellite",
            "--satellite_model", sat_model,
        ])

if RUN_STREET:
    for street_model in STREET_MODELS:
        run(base_cmd + [
            "--branch", "street",
            "--street_model", street_model,
            "--pooling", "mean",
        ])

if RUN_FUSION:
    for fusion_type in FUSION_TYPES:
        run(base_cmd + [
            "--branch", "fusion",
            "--satellite_model", FUSION_SAT_MODEL,
            "--street_model", FUSION_STREET_MODEL,
            "--fusion_type", fusion_type,
            "--pooling", "mean",
        ])

if RUN_EXPLAIN:
    explain_ckpt_dir = Path(DATA_ROOT) / "Results" / "global_learned" / "gdp"
    print("Set RUN_EXPLAIN after you have a finished checkpoint under:", explain_ckpt_dir)

print("Done. Learned-model outputs are under:")
print(Path(DATA_ROOT) / "Results" / "global_learned")
print("Check metrics.json, dataset_report.json, per_city_metrics.*, and val_predictions.csv inside each experiment folder.")
'''
a=100

In [54]:
'''!cd /content/CityLens && \
CITYLENS_DATA_ROOT=/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data \
python -m evaluate.global_learned.train \
  --task_name gdp \
  --epochs 10 \
  --batch_size 8 \
  --image_size 224 \
  --seed 42 \
  --num_workers 2 \
  --resume \
  --branch satellite \
  --satellite_model prithvi_rgb_lora'''

'!cd /content/CityLens && CITYLENS_DATA_ROOT=/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data python -m evaluate.global_learned.train   --task_name gdp   --epochs 10   --batch_size 8   --image_size 224   --seed 42   --num_workers 2   --resume   --branch satellite   --satellite_model prithvi_rgb_lora'

### 6b. Optional explainability runs

After a learned-model checkpoint exists, you can run explainability on multiple validation samples. Outputs are written under the experiment folder in `explain/`.

Available explainers:
- **Satellite**: integrated gradients with saved `.npy` attribution maps, input previews, and heatmaps
- **Street**: leave-one-view-out importance across validation samples
- **Fusion**: leave-one-modality-out ablation across validation samples

Feasibility note:
- These explainers are feasible with the current RGB-adapted pipeline.
- True multispectral spectral ablation is **not** feasible on CityLens and should remain framed as out of scope.

In [55]:
'''# Optional explainability example (edit CHECKPOINT_DIR before running)
import os
import subprocess
from pathlib import Path

DATA_ROOT = os.environ.get("CITYLENS_DATA_ROOT")
CHECKPOINT_DIR = ""  # e.g. /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/gdp/gdp-satellite-prithvi_rgb_lora-bs8-ep5-lr0.0002-seed42

if not CHECKPOINT_DIR:
    print("Set CHECKPOINT_DIR to a finished experiment folder, then run this cell.")
else:
    cmd = [
        "python", "-m", "evaluate.global_learned.explain",
        "--task_name", "gdp",
        "--branch", "satellite",
        "--satellite_model", "prithvi_rgb_lora",
        "--checkpoint_dir", CHECKPOINT_DIR,
    ]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd="/content/CityLens", check=True)'''

'# Optional explainability example (edit CHECKPOINT_DIR before running)\nimport os\nimport subprocess\nfrom pathlib import Path\n\nDATA_ROOT = os.environ.get("CITYLENS_DATA_ROOT")\nCHECKPOINT_DIR = ""  # e.g. /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/gdp/gdp-satellite-prithvi_rgb_lora-bs8-ep5-lr0.0002-seed42\n\nif not CHECKPOINT_DIR:\n    print("Set CHECKPOINT_DIR to a finished experiment folder, then run this cell.")\nelse:\n    cmd = [\n        "python", "-m", "evaluate.global_learned.explain",\n        "--task_name", "gdp",\n        "--branch", "satellite",\n        "--satellite_model", "prithvi_rgb_lora",\n        "--checkpoint_dir", CHECKPOINT_DIR,\n    ]\n    print("$", " ".join(cmd))\n    subprocess.run(cmd, cwd="/content/CityLens", check=True)'

In [56]:
'''!cd /content/CityLens && \
CITYLENS_DATA_ROOT=/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data \
python -m evaluate.global_learned.train \
  --task_name gdp \
  --epochs 20 \
  --batch_size 8 \
  --image_size 224 \
  --seed 42 \
  --num_workers 2 \
  --resume \
  --branch satellite \
  --satellite_model prithvi_rgb_lora \
  --lr 0.0002 \
  --backbone_lr 0.0002 \
  --head_lr 0.001'''

'!cd /content/CityLens && CITYLENS_DATA_ROOT=/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data python -m evaluate.global_learned.train   --task_name gdp   --epochs 20   --batch_size 8   --image_size 224   --seed 42   --num_workers 2   --resume   --branch satellite   --satellite_model prithvi_rgb_lora   --lr 0.0002   --backbone_lr 0.0002   --head_lr 0.001'

In [61]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data")
RESULTS_ROOT = DATA_ROOT / "Results" / "global_learned"

deleted = []
kept = []

for task_dir in RESULTS_ROOT.iterdir():
    if not task_dir.is_dir():
        continue

    # Collect all best.pt files under this task
    best_files = list(task_dir.glob("**/checkpoints/best.pt"))
    if not best_files:
        continue

    # Sort newest first by modification time
    best_files = sorted(best_files, key=lambda p: p.stat().st_mtime, reverse=True)

    keep = best_files[0]
    kept.append(str(keep))

    print(f"\nTask: {task_dir.name}")
    print(f"Keeping newest: {keep}")

    for old_best in best_files[1:]:
        try:
            old_best.unlink()
            deleted.append(str(old_best))
            print(f"Deleted older duplicate: {old_best}")
        except Exception as e:
            print(f"Failed to delete {old_best}: {e}")

print("\nSummary")
print(f"Kept {len(kept)} newest best.pt files")
print(f"Deleted {len(deleted)} older duplicate best.pt files")


Task: pop
Keeping newest: /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/pop/pop-satellite-prithvi_rgb_lora-clip_vitb16-bs8-ep5-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42/checkpoints/best.pt

Task: acc2health
Keeping newest: /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/acc2health/acc2health-satellite-prithvi_rgb_lora-clip_vitb16-bs8-ep30-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42/checkpoints/best.pt

Task: build_height
Keeping newest: /content/drive/MyDrive/GeoBench/CityLens-Data/CityLens-data/Results/global_learned/build_height/build_height-satellite-prithvi_rgb_lora-clip_vitb16-bs8-ep30-lr0.0002-blr0.0002-hlr0.001-ttlog1p-seed42/checkpoints/best.pt

Summary
Kept 3 newest best.pt files
Deleted 0 older duplicate best.pt files
